# Employee Score Model

This is the empolyee score model, used to calculate employee score/performance index. Firslty I will introduce all the parameters, affecting the score:

1. Task & Deliverable Performance
2. Quality of Work
3. Reliability & Consistency
4. Collaboration & Communication
5. Initiative, Ownership & Problem-Solving
6. Learning, Growth & Adaptability
7. Time & Resource Efficiency
8. Goal Alignment & Strategic Contribution
9. Leadership & Influence
10. Cultural Fit & Organisational Citizenship
11. Availability, Attendance & Workload Health
12. Client & Stakeholder Impact

As this model is to calculate score for different roles like:

1. Sales
2. Projects
3. Human Resources
4. Accounts
5. Team Managers
6. Vendor Manager

So, first 12 parameters here are same for all employees, then we have some domain specific parameters:

For **sales** we have:

13. Pipeline & Revenue Generation
14. Sales Process & CRM Discipline
15. Sales Relationships & Strategic Selling

For **projects** we have:

13. Project Delivery Excellence
14. Project Governance & Reporting
15. Stakeholder & Team Management

For **hr** we have:

13. Talent Acquisition & Workforce Planning
14. HR Operations & Compliance
15. People Experience & Strategic HR

For **accounts** we have:

13. Financial Accuracy & Reporting
14. Controls, Risk & Compliance
15. Stakeholder Service & Process Improvement

For **team managers** we have:

13. Team Outcomes & People Development
14. Operational Management
15. Culture, Inclusion & Strategic Leadership

For **vendor managers** we have:

13. Vendor Performance & Value
14. Procurement Process & Compliance
15. Relationship & Strategic Sourcing


Moving forward, more variables will be introduced within each parameter to capture every potential. I will then use a precise metric for each parameter calculation. Finally by weighting the parameters themselves, I will construct a robust, comprehensive scoring model that accounts for most of the edge cases also.

# Mathematical Concepts

I am defining all the mathematical concepts I used in this model:

## Beta-Adjusted Confidence Interval (BCI)

The problem: An employee who completed 1 out of 1 tasks has a 100% completion rate. An employee who completed 98 out of 100 has a 98% rate. Raw percentages say the first employee is better — but we have almost no evidence for that claim.

Instead of using the raw fraction (successes ÷ total), model the rate as a Beta distribution and extract the 5th percentile. This is the value you can be 95% confident the true rate is at least this high.

$$BCI(s,n)= B^{-1}(0.05, s+1, n-s+1)$$

Where:

- s= no of successes

- n= number of total attempts

- $B^{-1}$= inverse regularised incomplete Beta function.

|Scenario |Raw Rate|BCI Score|
| --- | --- | --- |
1 out of 1 |100%|~33%
10 out of 10|100%|~78%
50 out of 55|91%|~83%
98 out of 100|98%|~94%
980 out of 1000|98%|~97%

More data → the BCI converges toward the raw rate. 

Less data → heavy automatic penalty. 

This rewards employees with a solid, well-evidenced track record over those with a thin but technically perfect one.

<br>

## KDE Percentile Normalisation (KPN)

The problem: Manager A rates everyone 4-5 out of 5. Manager B rates everyone 2-3 out of 5. A rating of "3" from Manager A actually indicates poor performance; a "3" from Manager B is above average. Raw scores are not comparable across evaluators.

$$KPN_j(x)=\frac{1}{n} \sum_{i=1}^{n} \Phi\!\left( \frac{x - x_i}{h_j}\right)$$

Where:

- $x$ = the raw rating given to the employee
- $x_i$ = the $i$-th historical rating by evaluator $j$
- $\Phi$ = standard normal cumulative distribution function
- $h_j$ = bandwidth, calculated using Silverman's rule:

$$h_j=0.9\times\min\left(\hat{\sigma}_j,\frac{IQR_j}{1.34}\right)\times n_j^{-1/5}$$

Intuitive effect:

A "3/5" from a strict manager (whose average rating is 2.5) might become the 85th percentile → score ≈ 0.85

A "3/5" from a lenient manager (whose average rating is 4.0) might be the 20th percentile → score ≈ 0.20

Everyone is now on the same scale regardless of who evaluated them.

<br>

## Exponentially Weighted Moving Average (EWMA)

The problem An employee performed terribly two years ago but has been excellent for the last six months. Should old performance drag them down equally?


The decay parameter:

$$\lambda = 0.85$$

controls how quickly old data fades.


$$
\bar v=\frac{\sum_{t=1}^{T}\lambda^{T-t}v_t}{\sum_{t=1}^{T}\lambda^{T-t}}$$

$$\sigma_{EWMA}^{2}=(1-\lambda)\sum_{t=1}^{T}\lambda^{T-t}
\left(v_t-\bar v\right)^2$$

Where:

- $v_t$ = the measurement at time period $t$
- $T$ = the most recent period
- $\lambda = 0.85$

Intuitive effect with $\lambda = 0.85$:

| Period (most recent first) | Weight |
|------------|------------|
| Current month | 100% (reference) |
| 1 month ago | 85% |
| 2 months ago | 72% |
| 3 months ago | 61% |
| 6 months ago | 38% |
| 12 months ago | 14% |


An improving employee is rewarded; a declining one is penalised — even if their lifetime average is identical.

<br>

## Copula Multi-Rater Aggregation (CMA)

The problem: When multiple raters (manager, peer, tech lead) each assess the same trait, simply averaging their KPN-normalised scores ignores the fact that some raters tend to agree with each other.

Correlated raters should collectively carry less weight than independent ones.

Normalise each rater's score:

$$u_i = KPN_i(r_i)$$

Transform to standard normal:

$$z_i = \Phi^{-1}(u_i)$$

Estimate the correlation matrix $R$ from historical multi-rater data.

Compute the precision-weighted aggregate:

$$\bar z=\frac{\mathbf{1}^{T}R^{-1}}
{\mathbf{1}^{T}R^{-1}\mathbf{1}}$$

Back-transform:

$$CMA=\Phi(\bar z)$$

Intuitive effect:

If two raters always agree (correlation ≈ 1), they collectively count almost as one rater.

If three raters are completely independent (correlation ≈ 0), each carries full weight, and the aggregate is much more precise.


## Huber-Robust Sub-Composite (HRC)

The problem: When combining multiple variables into a sub-composite with a weighted average, a single extreme outlier variable can distort the entire composite.

For example, if one variable has a data entry error giving it a value of 0.01 while everything else is around 0.80, a simple weighted average drops significantly.



$$HRC=\arg\min_{\theta}\sum_i w_i\cdot\rho_\delta(x_i-\theta)$$

Where the Huber loss function is:

$$\rho_\delta(a)=\begin{cases}\frac{1}{2}a^2&|a|\le\delta\\[8pt]\delta\left(|a|-\frac{1}{2}\delta\right)&|a|>\delta\end{cases}$$

With

$$\delta = 0.15$$

### How to solve it (IRLS — Iteratively Reweighted Least Squares)

Start with the ordinary weighted average:

$$\theta^{(0)}=\sum w_i x_i$$

Update weights to down-weight outliers:

$$w_i^{(t)}=w_i\times\min\left(1,\frac{\delta}{|x_i-\theta^{(t)}|}\right)$$

Recompute the weighted average with updated weights:

$$\theta^{(t+1)}=\frac{\sum w_i^{(t)}x_i}{\sum w_i^{(t)}}$$

Repeat until:

$$\left|\theta^{(t+1)}-\theta^{(t)}\right|<
10^{-6}$$

Intuitive effect: 

Variables within 0.15 of the composite centre contribute at full weight.

Variables farther away have their influence reduced proportionally.

This makes the composite resistant to single-variable data errors.


## Logarithmic Normalisation (LN)

The problem: Count-based variables (number of initiatives, certifications, client mentions) have diminishing returns.

Going from 0 to 3 initiatives is a meaningful signal of proactivity.

Going from 30 to 33 is noise.

Apply a natural logarithm transformation with a saturation cap:

$$LN(x,c)=\min\left(\frac{\ln(1+x)}{\ln(1+c)},1\right)$$

Where $c$ is the saturation point (the count at which the score maxes out).

Intuitive effect (with $c=10$):

| Raw Count | LN Score |
|------------|------------|
| 0 | 0.00 |
| 1 | 0.29 |
| 3 | 0.58 |
| 5 | 0.75 |
| 10 | 1.00 |
| 20 | 1.00 (capped) |

The first few count a lot.

Additional ones matter less and less.


## Exponential Decay Penalty (EDP)

The problem: For variables where higher values indicate worse outcomes (days late, defects per deliverable, response time), the damage from the first unit is much greater than from additional units.


Being 1 day late on a deliverable is a serious issue.

Being 10 days late vs. 11 days late — the marginal damage is much smaller because the harm is already done.


$$EDP(x,k)=e^{-k x}$$

Where $k$ controls the decay rate.

Intuitive effect (with $k=0.15$ for delays):

| Days Late | EDP Score |
|------------|------------|
| 0 | 1.00 |
| 2 | 0.74 |
| 5 | 0.47 |
| 10 | 0.22 |
| 20 | 0.05 |










In [1]:
import numpy as np
import pandas as pd

from scipy.stats import beta
from scipy.stats import norm

def bci(successes, attempts, confidence=0.95):
    if attempts <= 0:
        return np.nan
    alpha = successes + 1
    beta_param = attempts - successes + 1
    percentile = 1 - confidence
    return beta.ppf(percentile, alpha, beta_param)


def silverman_bandwidth(history):
    history = np.asarray(history)
    n = len(history)
    sigma = np.std(history, ddof=1)
    q75, q25 = np.percentile(history, [75, 25])
    iqr = q75 - q25
    return 0.9 * min(sigma, iqr / 1.34) * (n ** (-1/5))

def kpn(raw_score, historical_scores):
    historical_scores = np.asarray(historical_scores)
    n = len(historical_scores)
    h = silverman_bandwidth(historical_scores)
    if h <= 0:
        return 0.5
    z = (raw_score - historical_scores) / h
    percentile = np.mean(norm.cdf(z))
    return percentile
    

def ewma_mean(values, lam=0.85):
    values = np.asarray(values)
    T = len(values)
    weights = np.array([lam ** (T - t - 1)for t in range(T)])
    weights /= weights.sum()
    return np.sum(weights * values)


def ewma_variance(values, lam=0.85):
    values = np.asarray(values)
    T = len(values)
    mean = ewma_mean(values, lam)
    weights = np.array([lam ** (T - t - 1)for t in range(T)])
    return (1 - lam) * np.sum(weights * (values - mean) ** 2)


def copula_multi_rater_aggregate(percentile_scores, correlation_matrix):
    u = np.clip(percentile_scores,1e-6,1 - 1e-6)
    z = norm.ppf(u)
    R_inv = np.linalg.inv(correlation_matrix)
    ones = np.ones(len(z))
    numerator = ones.T @ R_inv @ z
    denominator = ones.T @ R_inv @ ones
    z_bar = numerator / denominator
    return norm.cdf(z_bar)


def huber_robust_composite(values, weights=None,delta=0.15,tol=1e-6,max_iter=100):
    values = np.asarray(values)
    n = len(values)
    if weights is None:
        weights = np.ones(n)
    weights = np.asarray(weights)
    theta = np.average(values, weights=weights)
    for _ in range(max_iter):
        residuals = np.abs(values - theta)
        adjusted_weights = weights * np.minimum(1, delta / np.maximum(residuals,1e-12))
        theta_new = np.average( values,weights=adjusted_weights)
        if abs(theta_new - theta) < tol:
            break
        theta = theta_new
    return theta


def logarithmic_normalization(x,saturation_point):
    score = (np.log(1 + x)/np.log(1 + saturation_point))
    return min(score, 1.0)


def exponential_decay_penalty(x, k):
    return np.exp(-k * x)

In [2]:
from scipy.stats import beta as beta_dist
import numpy as np

def bci(successes, trials, confidence=0.05):
    if trials == 0:
        return 0.0
    s = successes if isinstance(successes, (int, float)) else successes
    n = trials
    return beta_dist.ppf(confidence, s + 1, n - s + 1)

def bci_from_rate(rate, assumed_n=50):
    s = round(rate * assumed_n)
    return bci(s, assumed_n)

def edp(x, k):
    return np.exp(-k * x)

def ln_norm(count, cap):
    if cap <= 0:
        return 0.0
    return min(np.log(1 + count) / np.log(1 + cap), 1.0)

def gaussian_penalty(actual, ideal, sigma):
    return np.exp(-((actual - ideal) ** 2) / (2 * sigma ** 2))

def hrc(values, weights, delta=0.15, max_iter=100, tol=1e-6):
    values = np.array(values, dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum() 
    theta = np.dot(weights, values)
    for _ in range(max_iter):
        residuals = np.abs(values - theta)
        adjusted_weights = weights.copy()
        for i in range(len(values)):
            if residuals[i] > delta:
                adjusted_weights[i] = weights[i] * (delta / residuals[i])
        if adjusted_weights.sum() == 0:
            break
        theta_new = np.dot(adjusted_weights, values) / adjusted_weights.sum()
        
        if abs(theta_new - theta) < tol:
            theta = theta_new
            break
        theta = theta_new
    return theta


In [3]:
data = [

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 1 — TASK & DELIVERABLE PERFORMANCE
    # "Does this employee get their work done, on time, and right?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many tasks did the employee complete on time, out of all tasks assigned?
        # Example: completed 44 out of 50 tasks on time → 0.88
        # [0.0 to 1.0]
        "tc": 0.88,

        # How many tasks did the employee actually complete in this evaluation period? (just the count)
        # [whole number, 0 or more]
        "tvnum": 52,

        # How many tasks is a person in this role EXPECTED to complete in the same period? (the benchmark/target)
        # [whole number, 1 or more]
        "tvbench": 45,

        # How many deliverables were submitted on time, out of all deliverables?
        # Example: 17 out of 20 deliverables on time → 0.85
        # [0.0 to 1.0]
        "dd": 0.85,

        # Total number of deliverables submitted.
        # [whole number, 1 or more]
        "tot_del": 20,

        # For the deliverables that WERE late, how many days late were they on average?
        # If nothing was late, enter 0.
        # Example: 3 late deliverables, averaging 2.5 days late → 2.5
        # [0 or more, can be decimal like 1.5]
        "adraw": 2.5,

        # Out of all HIGH-PRIORITY tasks, how many were completed on time?
        # Example: 9 out of 10 high-priority tasks done on time → 0.90
        # [0.0 to 1.0]
        "pr": 0.90,

        # How many tasks had to be ESCALATED to someone else (manager, senior, etc.) because the employee couldn't handle them, out of total tasks?
        # Example: 4 escalated out of 50 total → 0.08
        # [0.0 to 1.0] — lower is better
        "escraw": 0.08,

        # How many deliverables had to be sent back for REWORK (redo / major corrections), out of total completed?
        # Example: 5 reworked out of 50 completed → 0.10
        # [0.0 to 1.0] — lower is better
        "rwraw": 0.10,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 2 — QUALITY OF WORK
    # "How good is the actual output this employee produces?"
    # ═══════════════════════════════════════════════════════════════
    {
        # Overall quality rating given by manager / tech lead / peer reviewer, combined into one score.
        # 0 = terrible quality, 1 = exceptional quality
        # If you have a rating out of 5, divide by 5.
        # Example: 4.1 out of 5 → 0.82
        # [0.0 to 1.0]
        "qa": 0.82,

        # How many submissions were accepted on the FIRST TRY (no revisions needed), out of total submissions?
        # Example: 39 accepted first time out of 50 → 0.78
        # [0.0 to 1.0]
        "fa": 0.78,

        #Total number of submissions made.
        # [whole number, 1 or more]
        "tot_sub": 50,

        # On average, how many errors / defects / bugs were found per deliverable?
        # Example: 60 defects across 50 deliverables → 1.2
        # [0 or more, can be decimal]
        "drraw": 1.2,

        # How well does the employee follow standards, processes, SOPs, coding guidelines, or regulatory requirements?
        # 0 = ignores all standards, 1 = perfect compliance
        # [0.0 to 1.0]
        "cs": 0.85,

        # How thorough and detailed is the work?
        # Does the employee cover edge cases, think things through?
        # 0 = superficial / sloppy, 1 = exceptionally thorough
        # [0.0 to 1.0]
        "dt": 0.80,

        # How creative or innovative is the employee in solving problems? 
        # Does the employee come up with new approaches?
        # 0 = purely follows instructions, 1 = highly innovative
        # If creativity is NOT relevant to this role, enter 0.50
        # [0.0 to 1.0]
        "cr": 0.65,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 3 — RELIABILITY & CONSISTENCY
    # "Can you count on this person to deliver predictably,
    #  month after month, without babysitting?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How much does this employee's performance FLUCTUATE from month to month?
        # This is the standard deviation of their monthly scores.
        # 0.00 = perfectly consistent every month
        # 0.05 = very consistent
        # 0.15 = moderate ups and downs
        # 0.30 = wildly inconsistent
        # If you don't know, estimate based on gut feel.
        # [0.0 to 0.50 typically]
        "cvsigma": 0.12,

        # How much does the manager TRUST this employee to deliver without checking in?
        # 0 = no trust at all, 1 = complete trust
        # [0.0 to 1.0]
        "dp": 0.85,

        # When this employee says "I'll do it", how often do they actually follow through and complete it?
        # Example: followed through on 18 out of 20 commitments → 0.90
        # [0.0 to 1.0]
        "fs": 0.90,

        # Total number of commitments made by the employee.
        # # [whole number, 1 or more]
        "tot_com": 20,

        # During critical crunch periods (deadlines, launches, emergencies), how often was this employee available and present?
        # Example: available for 4 out of 5 critical periods → 0.80
        # [0.0 to 1.0]
        "av": 0.80,

        # Total number of critical periods faced from when the employee joined.
        # [whole number, 1 or more]
        "tot_cri": 5,

        # Pick a representative task. How many days did the employee ESTIMATE it would take?
        # [any positive number, can be decimal]
        "pmest": 5.0,

        # For that same task, how many days did it ACTUALLY take?
        # [any positive number, can be decimal]
        "pmactual": 6.0,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 4 — COLLABORATION & COMMUNICATION
    # "How well does this person work with others and share info?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How much does this employee contribute to TEAM goals (not just their own tasks)?
        # Rated by peers.
        # 0 = works in isolation, 1 = exceptional team player
        # [0.0 to 1.0]
        "tc4": 0.80,

        # How clearly does this employee communicate? (emails, meetings, documentation, presentations)
        # 0 = confusing and unclear, 1 = crystal clear communicator
        # [0.0 to 1.0]
        "cm": 0.85,

        # On average, how many HOURS does it take for this employee to respond to messages, emails, or requests from colleagues?
        # Example: usually replies within 3 hours → 3.0
        # [0 or more, can be decimal like 1.5]
        "rsphours": 3.0,

        # How well does this employee handle disagreements and conflicts at work?
        # 0 = creates or escalates conflict, 1 = resolves issues constructively
        # [0.0 to 1.0]
        "cf": 0.75,

        # How well does this employee share knowledge with others?
        # (writing docs, mentoring juniors, giving presentations, helping teammates learn)
        # 0 = hoards knowledge, 1 = actively shares and teaches
        # [0.0 to 1.0]
        "kn": 0.70,

        # How well does this employee RECEIVE feedback?
        # Do they listen, accept, and actually improve based on it?
        # 0 = defensive / ignores feedback, 1 = embraces and acts on it
        # [0.0 to 1.0]
        "fb": 0.80,

        # Total number of handoffs occured.
        # [whole number, 1 or more]
        "tot_han": 60,
        
        # How often does this employee BLOCK others from doing their work?
        # (holding up reviews, not responding to handoffs, being a bottleneck)
        # Example: blocked others 3 times out of 60 handoff
        # opportunities → 3/60 = 0.05
        # [0.0 to 1.0] — lower is better
        "blraw": 0.05,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 5 — INITIATIVE, OWNERSHIP & PROBLEM-SOLVING
    # "Does this employee go beyond what's assigned and take
    #  ownership of outcomes?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many improvements, ideas, or initiatives did the employee START ON THEIR OWN (not assigned by anyone) during this evaluation period?
        # Example: suggested and implemented 4 improvements → 4
        # [whole number, 0 or more]
        "incount": 4,

        # How much ownership does this employee take?
        # Do they see things through from start to finish, or hand off problems to others?
        # 0 = avoids responsibility, 1 = full end-to-end ownership
        # [0.0 to 1.0]
        "ow": 0.85,

        # How good is this employee at solving problems?
        # Do they find ROOT CAUSES, or just apply band-aids?
        # 0 = can't solve problems independently,
        # 1 = excellent analytical problem-solver
        # [0.0 to 1.0]
        "ps": 0.80,

        # How independently can this employee work?
        # How much supervision do they need?
        # 0 = needs constant hand-holding,
        # 1 = fully self-directed
        # [0.0 to 1.0]
        "au": 0.82,

        # How many times did the employee contribute meaningfully to ANOTHER TEAM's work? (cross-functional contributions)
        # Example: helped 2 other teams → 2
        # [whole number, 0 or more]
        "bccount": 2,

        # Did the employee's self-initiated improvements actually produce measurable results?
        # 0 = no real impact, 1 = significant measurable outcomes
        # [0.0 to 1.0]
        "imp": 0.70,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 6 — LEARNING, GROWTH & ADAPTABILITY
    # "Is this employee growing and adapting, or standing still?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many NEW SKILLS or CERTIFICATIONS did the employee acquire during this period?
        # (courses completed, certs earned, new tools mastered)
        # [whole number, 0 or more]
        "skcount": 2,

        # How actively does the employee pursue professional development? (attending training, reading, conferences, self-study)
        # 0 = zero interest in development,
        # 1 = actively and consistently developing
        # [0.0 to 1.0]
        "pd": 0.70,

        # How well does the employee adapt to CHANGE?
        # (new tools, new processes, shifting requirements, team restructuring)
        # 0 = resists all change, 1 = thrives on change
        # [0.0 to 1.0]
        "ad6": 0.80,

        # In the LAST evaluation, were areas for improvement identified? If so, how much did the employee actually improve in those areas?
        # 0 = no improvement at all,
        # 1 = fully addressed all previous feedback
        # If this is the first evaluation, enter 0.50
        # [0.0 to 1.0]
        "fp": 0.75,

        # How quickly does this employee become productive in
        # NEW areas compared to peers?
        # 0 = very slow learner, 1 = picks things up immediately
        # [0.0 to 1.0]
        "lr": 0.70,

        # How many DIFFERENT ROLES or FUNCTIONS can this employee competently perform?
        # Example: can do their own role + 2 adjacent ones → 3
        # [whole number, 1 or more]
        "vrroles": 2,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 7 — TIME & RESOURCE EFFICIENCY
    # "Is this employee productive with their time and resources?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What fraction of available work hours are actually spent on productive work?
        # Example: 6.8 productive hours out of 8 available → 0.85
        # [0.0 to 1.0]
        "ut": 0.85,

        # How much output does this employee produce per unit of effort, compared to role benchmarks?
        # 0 = far below benchmark, 1 = at or above benchmark
        # [0.0 to 1.0]
        "et": 0.80,

        # How efficiently does the employee use company resources?
        # (budget, tools, software licenses, assets)
        # 0 = wasteful, 1 = very efficient
        # [0.0 to 1.0]
        "rt": 0.75,

        # Total number of meetings happened.
        # [whole number, 1 or more]
        "tot_meet": 20,

        # Out of all meetings attended, how many were actually productive and valuable?
        # Example: 14 productive meetings out of 20 total → 0.70
        # [0.0 to 1.0]
        "mt": 0.70,

        # What fraction of this employee's hours are OVERTIME, compared to standard hours?
        # Example: 4 overtime hours per 40 standard hours → 0.10
        # NOTE: Some overtime is normal. Excessive overtime signals inefficiency or unsustainable workload.
        # [0.0 or more, typically 0.0 to 0.50]
        "otratio": 0.15,

        # What fraction of total work hours were WASTED or IDLE?
        # (non-productive time, waiting, unnecessary rework, unproductive browsing)
        # Example: 1 hour wasted out of 8 → 0.125
        # [0.0 to 1.0] — lower is better
        "wmraw": 0.12,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 8 — GOAL ALIGNMENT & STRATEGIC CONTRIBUTION
    # "Is this employee's work advancing what the company
    #  actually cares about?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What fraction of the employee's tasks are directly linked to company goals / OKRs / KPIs?
        # Example: 42 out of 50 tasks mapped to OKRs → 0.84
        # [0.0 to 1.0]
        "ga": 0.85,

        # Total number of tasks assigned.
        # [whole number, 1 or more]
        "tot_task": 50,

        # Total number of individual goals assigned.
        # [whole number, 1 or more]
        "tot_goal": 5,

        # What fraction of the employee's INDIVIDUAL GOALS were met or exceeded?
        # Example: hit 4 out of 5 goals → 0.80
        # [0.0 to 1.0]
        "gc": 0.80,

        # How well does the employee UNDERSTAND the company's strategy and connect their daily work to it?
        # 0 = no awareness, 1 = deeply strategic thinker
        # [0.0 to 1.0]
        "st": 0.75,

        # Does the employee consistently work on the MOST IMPORTANT things first?
        # 0 = poor prioritisation, 1 = always works on highest impact
        # [0.0 to 1.0]
        "pv": 0.78,

        # Has the employee's work had a VISIBLE, MEASURABLE impact on team or business metrics?
        # 0 = no visible impact, 1 = clearly moved the needle
        # [0.0 to 1.0]
        "vi": 0.72,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 9 — LEADERSHIP & INFLUENCE
    # "Does this person make the people around them better?
    #  (with or without a manager title)"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many people is the employee actively MENTORING or coaching (formally or informally)?
        # [whole number, 0 or more]
        "mncount": 2,

        # How much INFLUENCE does this employee have on others?
        # Do people adopt their ideas, follow their standards, look to them for guidance?
        # 0 = no influence, 1 = widely respected thought leader
        # [0.0 to 1.0]
        "inf": 0.70,

        # How effectively does this employee DELEGATE work to others?
        # If the employee has NO direct reports and doesn't delegate, enter 0.50 (neutral — not applicable)
        # 0 = terrible at delegating, 1 = excellent delegation
        # [0.0 to 1.0]
        "de": 0.50,

        # How good are the DECISIONS this employee makes within their authority?
        # 0 = poor judgment, 1 = consistently excellent decisions
        # [0.0 to 1.0]
        "dm": 0.80,

        # Do the people who work WITH or AROUND this employee perform BETTER because of them?
        # 0 = drags the team down, 1 = significantly elevates team
        # [0.0 to 1.0]
        "te": 0.75,

        # How does this employee perform UNDER PRESSURE?
        # (during crises, incidents, tight deadlines, emergencies)
        # 0 = falls apart, 1 = thrives and leads through crisis
        # [0.0 to 1.0]
        "cr9": 0.72,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 10 — CULTURAL FIT & ORGANISATIONAL CITIZENSHIP
    # "Does this employee align with company values and
    #  contribute to a positive workplace?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How well does the employee's behaviour align with the company's stated values?
        # 0 = actively contradicts values, 1 = embodies them fully
        # [0.0 to 1.0]
        "va": 0.85,

        # How ethical and honest is this employee?
        # (integrity, transparency, trustworthiness)
        # 0 = dishonest / unethical, 1 = impeccable integrity
        # [0.0 to 1.0]
        "eth": 0.90,

        # Does this employee contribute to a POSITIVE work environment? (supportive, inclusive, constructive)
        # 0 = toxic presence, 1 = everyone's better off with them
        # [0.0 to 1.0]
        "po": 0.80,

        # How many VOLUNTARY, non-mandatory activities did the employee participate in?
        # (social events, committees, charity drives, team outings, culture initiatives)
        # [whole number, 0 or more]
        "evcount": 3,

        # How many CONFIRMED negative behavioural incidents were recorded? 
        # (HR complaints, policy violations, disciplinary actions)
        #  WARNING: 5 or more incidents = this entire parameter
        #             score drops to ZERO.
        # [whole number, 0 or more] — lower is better
        "ngcount": 0,

        # How professional is this employee's general conduct?
        # (punctuality to meetings, email etiquette, dress code, respectful communication)
        # 0 = unprofessional, 1 = exemplary professionalism
        # [0.0 to 1.0]
        "prc": 0.88,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 11 — AVAILABILITY, ATTENDANCE & WORKLOAD HEALTH
    # "Is the employee present, and is their work pattern
    #  sustainable?"
    # ═══════════════════════════════════════════════════════════════
    {
        # Out of total working days (AFTER removing approved leave), how many days was the employee actually present?
        # Example: present 190 out of 200 working days → 0.95
        # [0.0 to 1.0]
        "at": 0.95,

        # Out of total working days, how many days did the employee arrive ON TIME?
        # Example: on time 180 out of 200 days → 0.90
        # [0.0 to 1.0]
        "pu": 0.90,

        # What fraction of working days were UNPLANNED absences?
        # (calling in sick last minute, no-shows, unexplained absences)
        # Example: 10 unplanned absence days out of 200 → 0.05
        # [0.0 to 1.0] — lower is better
        "uaraw": 0.05,

        # How much BURNOUT RISK does this employee show?
        # Consider: overtime trends, unused leave, engagement scores, self-reported stress, withdrawal from team activities.
        # 0.00 = no burnout signs at all
        # 0.20 = mild signs
        # 0.50 = moderate burnout risk
        # 0.80 = severe burnout risk
        # 1.00 = critical burnout
        # [0.0 to 1.0] — lower is better
        "bn": 0.20,

        # What is the employee's ACTUAL workload compared to the IDEAL workload?
        # 1.00 = perfectly balanced (doing exactly the right amount)
        # 0.70 = underloaded (only doing 70% of ideal capacity)
        # 1.30 = overloaded (doing 130% of ideal capacity)
        # [0.5 to 2.0 typically, 1.0 is ideal]
        "wlratio": 1.15,

        # Total number of woking days for that employee.
        # [whole number, 1 or more]
        "tot_days": 200,
        
        # What fraction of their available leave did the employee actually TAKE?
        # Example: took 16 leave days out of 200 working days → 0.08
        # [0.0 to 1.0]
        "lvratio": 0.07,

        # What is the company's TARGET leave ratio?
        # (how much leave does the company WANT employees to take?)
        # Example: company expects ~8% of working days as leave → 0.08
        # [0.0 to 1.0]
        "lvtarget": 0.08,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 12 — CLIENT & STAKEHOLDER IMPACT
    # "What value did this employee deliver to clients or
    #  internal stakeholders?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How satisfied are clients / stakeholders with this employee's work specifically?
        # 0 = very dissatisfied, 1 = extremely satisfied
        # If you have a rating out of 5, divide by 5.
        # Example: average 4.1 out of 5 → 0.82
        # [0.0 to 1.0]
        "csat": 0.82,

        # How many times did a client or stakeholder give POSITIVE feedback specifically mentioning this employee?
        # (thank-you emails, commendations, shout-outs)
        # [whole number, 0 or more]
        "cfbcount": 5,

        # Total number of client accounts, this employee worked on.
        # [whole number, 1 or more]
        "tot_acc": 20,
        
        # Out of all client accounts this employee worked on, how many were RETAINED or EXPANDED?
        # Example: 17 retained out of 20 total → 0.85
        # If not applicable (no client accounts), enter 0.50
        # [0.0 to 1.0]
        "ret": 0.85,

        # How much revenue is DIRECTLY ATTRIBUTABLE to this employee's work? (in your currency)
        # If not applicable or not tracked, enter the same value as the target below (so the ratio = 1.0, neutral)
        # [any number, 0 or more]
        "revattributed": 1100000,

        # What was the revenue TARGET for this employee?
        # [any number, 1 or more]
        "revtarget": 1000000,

        # How many NEGATIVE client interactions were recorded?
        # (complaints, escalations from clients, formal grievances)
        # WARNING: 7 or more incidents significantly tanks this score.
        # [whole number, 0 or more] — lower is better
        "nccount": 1,

        # How good is this employee at building lasting professional RELATIONSHIPS with clients / stakeholders beyond just the current project?
        # 0 = purely transactional, 1 = builds deep, lasting trust
        # [0.0 to 1.0]
        "re": 0.75,
    },
]

gates=['f','f','f','f','f','f','f']
modifiers = [1.0,1.0,1.0,0.9,1.0]

In [4]:
p1 = data[0] 

Tc = bci(p1["tvnum"], round( p1["tvnum"]/ p1["tc"]))

Tv = min(p1["tvnum"] / p1["tvbench"], 1.3) / 1.3
assumed_deliverables = p1["tot_del"]
Dd = bci(round(p1["dd"] * assumed_deliverables), assumed_deliverables)
Ad = edp(p1["adraw"], 0.15)
assumed_hp = 10
Pr = bci(round(p1["pr"] * assumed_hp), assumed_hp)
esc_count = round(p1["escraw"] * p1["tvnum"])
Esc = 1.0 - bci(esc_count, p1["tvnum"])
rw_count = round(p1["rwraw"] * p1["tvnum"])
Rw = 1.0 - bci(rw_count, p1["tvnum"])
T_core = hrc(
    values=[Tc, Dd, Ad, Pr],
    weights=[0.30, 0.25, 0.20, 0.25]
)

T_drag = hrc(
    values=[Esc, Rw],
    weights=[0.50, 0.50]
)

X1 = 0.60 * T_core + 0.20 * Tv + 0.20 * T_drag

print(X1)

0.7899407076108276


In [5]:
p2 = data[1]

Qa = p2["qa"]  

assumed_submissions = p2["tot_sub"]
Fa = bci(round(p2["fa"] * assumed_submissions), assumed_submissions)
Dr = edp(p2["drraw"], 0.4)
Cs = p2["cs"]   
Dt = p2["dt"] 
Cr = p2["cr"] 

Q_out = hrc(
    values=[Qa, Fa, Dr, Cs, Dt],
    weights=[0.25, 0.20, 0.20, 0.15, 0.20]
)

X2 = 0.85 * Q_out + 0.15 * Cr

print(X2)

0.7349355257598978


In [6]:
p3 = data[2]

Cv = np.exp(-2 * np.sqrt(p3["cvsigma"] ** 2))
Dp = p3["dp"]  

assumed_commitments = p3["tot_com"]
Fs = bci(round(p3["fs"] * assumed_commitments), assumed_commitments)


assumed_critical = p3["tot_cri"]
Av = bci(round(p3["av"] * assumed_critical), assumed_critical)

Pm = 1.0 - min(abs(p3["pmactual"] - p3["pmest"]) / p3["pmest"], 1.0)

X3 = hrc(
    values=[Cv, Dp, Fs, Av, Pm],
    weights=[0.25, 0.25, 0.25, 0.10, 0.15]
)

print(X3)

0.7739100410124575


In [7]:
p4 = data[3]

Tc4 = p4["tc4"]  

Cm = p4["cm"]     

Rsp = edp(p4["rsphours"], 0.05)

Cf = p4["cf"]   

Kn = p4["kn"]    
Fb = p4["fb"]      

assumed_handoffs = p4["tot_han"]
bl_count = round(p4["blraw"] * assumed_handoffs)
Bl = 1.0 - bci(bl_count, assumed_handoffs)

C_pos = hrc(
    values=[Tc4, Cm, Rsp, Cf, Kn, Fb],
    weights=[0.25, 0.20, 0.15, 0.10, 0.15, 0.15]
)

X4 = C_pos * Bl

print(X4)

0.7809611074902966


In [8]:
p5 = data[4]

In = ln_norm(p5["incount"], 10)
Ow = p5["ow"]     

Ps = p5["ps"]    

Au = p5["au"]    

Bc = ln_norm(p5["bccount"], 5)

Imp = p5["imp"]  

X5 = hrc(
    values=[In, Ow, Ps, Au, Bc, Imp],
    weights=[0.20, 0.25, 0.20, 0.15, 0.10, 0.10]
)

print(X5)

0.7610522675707938


In [9]:
p6 = data[5]

Sk = ln_norm(p6["skcount"], 5)

Pd = p6["pd"]      
Ad6 = p6["ad6"]  
Fp = p6["fp"]   
Lr = p6["lr"]   

Vr = min(p6["vrroles"] / 3.0, 1.0)

X6 = hrc(
    values=[Sk, Pd, Ad6, Fp, Lr, Vr],
    weights=[0.20, 0.15, 0.25, 0.20, 0.10, 0.10]
)

print(X6)


0.7142961052197584


In [10]:
p7 = data[6]

Ut = p7["ut"]     

Et = p7["et"]  

Rt = p7["rt"] 

assumed_meetings = p7["tot_meet"]
Mt = bci(round(p7["mt"] * assumed_meetings), assumed_meetings)
Ot = edp(max(0, p7["otratio"] - 0.10), 2.0)

Wm = 1.0 - p7["wmraw"]


X7 = hrc(
    values=[Ut, Et, Rt, Mt, Ot, Wm],
    weights=[0.25, 0.25, 0.15, 0.10, 0.15, 0.10]
)

print(X7)

0.8152506745557503


In [11]:
p8 = data[7]

assumed_tasks_8 = p8["tot_task"]
Ga = bci(round(p8["ga"] * assumed_tasks_8), assumed_tasks_8)

assumed_goals = p8["tot_goal"]
Gc = bci(round(p8["gc"] * assumed_goals), assumed_goals)

St = p8["st"]   

Pv = p8["pv"]

Vi = p8["vi"] 
X8 = hrc(
    values=[Ga, Gc, St, Pv, Vi],
    weights=[0.25, 0.30, 0.15, 0.15, 0.15]
)

print(X8)

0.6802587131672205


In [12]:
p9 = data[8]

Mn = ln_norm(p9["mncount"], 4)

Inf = p9["inf"] 

De = p9["de"]   

Dm = p9["dm"]     

Te = p9["te"]  

Cr9 = p9["cr9"] 

X9 = hrc(
    values=[Mn, Inf, De, Dm, Te, Cr9],
    weights=[0.15, 0.20, 0.15, 0.20, 0.20, 0.10]
)

print(X9)

0.7081069567145918


In [13]:
p10 = data[9]

Va = p10["va"]

Eth = p10["eth"]

Po = p10["po"]

Ev = ln_norm(p10["evcount"], 6)

Ng = max(0, 1.0 - 0.2 * p10["ngcount"])

Prc = p10["prc"] 

C_cit = hrc(
    values=[Va, Eth, Po, Ev, Prc],
    weights=[0.25, 0.25, 0.20, 0.15, 0.15]
)

X10 = Ng * C_cit

print(X10)

0.8363621561324067


In [14]:
p11 = data[10]

assumed_workdays = p11["tot_days"]
At = bci(round(p11["at"] * assumed_workdays), assumed_workdays)

Pu = bci(round(p11["pu"] * assumed_workdays), assumed_workdays)

ua_count = round(p11["uaraw"] * assumed_workdays)
Ua = 1.0 - bci(ua_count, assumed_workdays)

Bn = 1.0 - p11["bn"]

Wl = gaussian_penalty(p11["wlratio"], 1.0, 0.20)

Lv = gaussian_penalty(p11["lvratio"], p11["lvtarget"], 0.15)

X11 = hrc(
    values=[At, Pu, Ua, Bn, Wl, Lv],
    weights=[0.25, 0.15, 0.20, 0.15, 0.15, 0.10]
)

print(X11)

0.8848790119626456


In [15]:
p12 = data[11]

Csat = p12["csat"] 

Cfb = ln_norm(p12["cfbcount"], 10)

assumed_accounts = p12["tot_acc"]
Ret = bci(round(p12["ret"] * assumed_accounts), assumed_accounts)

Rev = min(p12["revattributed"] / p12["revtarget"], 1.2) / 1.2

Nc = max(0, 1.0 - 0.15 * p12["nccount"])

Re = p12["re"]   

S_impact = hrc(
    values=[Csat, Cfb, Ret, Rev, Re],
    weights=[0.25, 0.15, 0.20, 0.20, 0.20]
)

X12 = Nc * S_impact

print(X12)

0.6668882314041245


In [16]:
role=input("The role of employee, you want to calculate your score: ").lower()
while role not in ["sales","projects","hr","accounts","teammanager","vendor_manager"]:
    role=input("Enter a valid role/domain: ").lower()

The role of employee, you want to calculate your score:  sales


In [17]:
if role == "sales":
    sales_data = [
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 13 — PIPELINE & REVENUE GENERATION
        # "Is this salesperson building enough pipeline and
        #  converting it into real revenue?"
        # ═══════════════════════════════════════════════════════════════
        {
            # What is the total pipeline value generated this period ($)?
            # [any number, 0 or more]
            "pv": 850000,
            # What is the pipeline target for this period ($)?
            # [any number, 1 or more]
            "pvtarget": 1000000,
            # How many leads were converted into qualified opportunities?
            # [whole number, 0 or more]
            "lconv": 28,
            # How many total leads were worked/touched this period?
            # [whole number, 1 or more]
            "lworked": 120,
            # How many deals were closed-won this period?
            # [whole number, 0 or more]
            "dclosed": 14,
            # What is the average deal size for this rep ($)?
            # [any number, 0 or more]
            "avgdeal": 62000,
            # What is the team benchmark average deal size ($)?
            # [any number, 1 or more]
            "avgdealbench": 55000,
            # What is the average number of days to close a deal for this rep?
            # [any positive number, can be decimal]
            "scdays": 38,
            # What is the benchmark/target days to close?
            # [any positive number, can be decimal]
            "scdaysbench": 35,
            # How much upsell/cross-sell revenue was generated ($)?
            # [any number, 0 or more]
            "uprev": 45000,
            # What is the total base closed revenue ($)?
            # [any number, 0 or more]
            "baserev": 870000,
            # How many accounts churned that are attributable to this rep?
            # [whole number, 0 or more] — lower is better
            "achurn": 2,
            # What revenue forecast did the rep submit ($)?
            # [any number, 0 or more]
            "fcsub": 900000,
            # What was the actual revenue realised ($)?
            # [any number, 0 or more]
            "fcactual": 870000,
            # What is the quota target for this rep ($)?
            # [any number, 1 or more]
            "quota": 800000,
            # How much revenue did the rep actually achieve ($)?
            # [any number, 0 or more]
            "quotaach": 870000,
            # How many brand-new customers (new logos) were acquired?
            # [whole number, 0 or more]
            "newlogos": 5,
            # What is the target for new logos?
            # [whole number, 1 or more]
            "newlogotgt": 4,
            # What is the weighted pipeline accuracy (weighted pipeline vs. actual close rate)?
            # 0 = completely inaccurate, 1 = perfectly calibrated
            # [0.0 to 1.0]
            "wpacc": 0.82,
            # What fraction of demos progressed to the proposal stage?
            # Example: 13 proposals from 20 demos → 0.65
            # [0.0 to 1.0]
            "d2p": 0.65,
            # What fraction of proposals became closed-won deals?
            # Example: 9 wins from 20 proposals → 0.45
            # [0.0 to 1.0]
            "pwin": 0.45,
            # What is the average discount percentage given by this rep?
            # Example: 12% average discount → 12.0
            # [0.0 to 100.0] — lower is generally better
            "discavg": 12.0,
            # What is the benchmark maximum discount percentage?
            # [0.0 to 100.0]
            "discbench": 15.0,
            # What fraction of total revenue comes from the single largest account?
            # Example: 35% from one account → 0.35
            # [0.0 to 1.0] — lower is better (less concentration risk)
            "revconc": 0.35,
            # How many deals included a multi-year commitment?
            # [whole number, 0 or more]
            "mydeals": 3,
            # What is the total number of deals (for calculating multi-year ratio)?
            # [whole number, 1 or more]
            "totaldeals": 14,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 14 — SALES PROCESS & CRM DISCIPLINE
        # "Does this salesperson follow the sales process, keep the
        #  CRM clean, and operate with rigour?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many CRM fields were filled in correctly?
            # [whole number, 0 or more]
            "crmcorr": 480,
            # How many total CRM fields were expected to be filled?
            # [whole number, 1 or more]
            "crmtotal": 520,
            # How many calls/meetings were made this period?
            # [whole number, 0 or more]
            "callsmade": 340,
            # What is the activity target for calls/meetings?
            # [whole number, 1 or more]
            "callstgt": 300,
            # What is the manager-rated quality score for proposals (out of 5)?
            # Example: 4.2 out of 5
            # [0.0 to 5.0]
            "propqual": 4.2,
            # Historical proposal quality ratings from the manager (list of scores out of 5).
            # Used to assess trend/consistency.
            # [list of numbers, each 0.0 to 5.0]
            "propqualhist": [3.5, 3.8, 4.0, 3.9, 4.1, 3.7, 4.3, 4.0],
            # How many deals had proper discovery documentation completed?
            # [whole number, 0 or more]
            "discdoc": 25,
            # How many total deals required discovery documentation?
            # [whole number, 1 or more]
            "disctotal": 28,
            # How many deals were lost due to rep mistakes (not price or fit)?
            # [whole number, 0 or more] — lower is better
            "dlostrerr": 2,
            # What fraction of pipeline stages were updated within 24 hours?
            # Example: 88% updated on time → 0.88
            # [0.0 to 1.0]
            "stageupd": 0.88,
            # What fraction of leads were followed up within SLA?
            # Example: 92% followed up on time → 0.92
            # [0.0 to 1.0]
            "fucomp": 0.92,
            # What is the post-sale handoff quality rating from the CS team?
            # 0 = terrible handoff, 1 = seamless handoff
            # [0.0 to 1.0]
            "handoff": 0.85,
            # How many competitive intelligence reports did the rep submit?
            # [whole number, 0 or more]
            "cintel": 4,
            # What fraction of the territory/account plan was adhered to?
            # 0 = completely off-plan, 1 = perfect adherence
            # [0.0 to 1.0]
            "tpadh": 0.80,
            # What fraction of quotes were accurate (no pricing/config errors)?
            # Example: 93% correct → 0.93
            # [0.0 to 1.0]
            "qtacc": 0.93,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 15 — SALES RELATIONSHIPS & STRATEGIC SELLING
        # "Is this salesperson building executive-level
        #  relationships and selling strategically?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many C-level/VP contacts does this rep have in the CRM?
            # [whole number, 0 or more]
            "execcon": 8,
            # What is the target for executive contacts?
            # [whole number, 1 or more]
            "execcontgt": 10,
            # How many customer referrals did this rep generate?
            # [whole number, 0 or more]
            "refgen": 3,
            # What is the Net Promoter Score from this rep's customers?
            # [-100 to 100]
            "nps": 42,
            # What is the average engagement score of managed accounts?
            # 0 = no engagement, 1 = deeply engaged
            # [0.0 to 1.0]
            "custeng": 0.78,
            # What fraction of strategic accounts have an active engagement plan?
            # Example: 85% covered → 0.85
            # [0.0 to 1.0]
            "stracov": 0.85,
            # How many deals had partner involvement?
            # [whole number, 0 or more]
            "partdeals": 2,
            # How many leads from events/webinars were converted?
            # [whole number, 0 or more]
            "evleadconv": 5,
            # How many total event leads were assigned to this rep?
            # [whole number, 1 or more]
            "evleadtot": 20,
            # How many thought leadership contributions did the rep make?
            # (blog posts, webinars, case studies authored)
            # [whole number, 0 or more]
            "tlcontrib": 2,
            # What fraction of accounts have an updated health score?
            # Example: 90% updated → 0.90
            # [0.0 to 1.0]
            "chupd": 0.90,
            # How many accounts have a formal expansion plan created?
            # [whole number, 0 or more]
            "expplan": 4,
            # What is the target for expansion plans?
            # [whole number, 1 or more]
            "expplantgt": 6,
        },
    ]
elif role == "projects":
    projects_data = [
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 13 — PROJECT DELIVERY EXCELLENCE
        # "Is this project manager delivering on scope, budget,
        #  and schedule with quality?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many deliverables remained within the original scope?
            # [whole number, 0 or more]
            "delscope": 42,
            # How many total deliverables were there?
            # [whole number, 1 or more]
            "deltotal": 48,
            # What was the actual project spend ($)?
            # [any number, 0 or more]
            "actspend": 285000,
            # What was the allocated budget ($)?
            # [any number, 1 or more]
            "budget": 300000,
            # What is the Schedule Performance Index (EVM)?
            # 1.0 = on schedule, >1 = ahead, <1 = behind
            # [0.0 or more, typically 0.5 to 1.5]
            "spi": 1.05,
            # What is the Cost Performance Index (EVM)?
            # 1.0 = on budget, >1 = under budget, <1 = over budget
            # [0.0 or more, typically 0.5 to 1.5]
            "cpi": 0.98,
            # How many milestones were delivered on time?
            # [whole number, 0 or more]
            "mshit": 11,
            # How many total milestones were there?
            # [whole number, 1 or more]
            "mstotal": 13,
            # How many risks were identified proactively (before they materialised)?
            # [whole number, 0 or more]
            "riskearly": 8,
            # How many total risks materialised during the project?
            # [whole number, 1 or more]
            "risktotal": 10,
            # What is the stakeholder satisfaction score (out of 5)?
            # [0.0 to 5.0]
            "stksat": 4.1,
            # Historical stakeholder satisfaction scores (list of scores out of 5).
            # Used to assess trend/consistency.
            # [list of numbers, each 0.0 to 5.0]
            "stksathist": [3.5, 3.8, 4.0, 3.6, 4.2, 3.9, 4.1],
            # How many change requests were raised during the project?
            # [whole number, 0 or more] — lower generally indicates better scope management
            "chgreq": 5,
            # How many items were in the original scope?
            # [whole number, 1 or more]
            "origscope": 48,
            # How many days was the project blocked due to unmanaged dependencies?
            # [whole number, 0 or more] — lower is better
            "blkdays": 4,
            # What is the total project duration in days?
            # [whole number, 1 or more]
            "projdays": 180,
            # How many defects were found at delivery/UAT?
            # [whole number, 0 or more] — lower is better
            "defects": 6,
            # How many total deliverables were at the delivery point?
            # [whole number, 1 or more]
            "deldel": 42,
            # How many items required rework post-delivery?
            # [whole number, 0 or more] — lower is better
            "rework": 3,
            # How many deliverables met all acceptance criteria?
            # [whole number, 0 or more]
            "accmet": 40,
            # How many total deliverables had acceptance criteria defined?
            # [whole number, 1 or more]
            "acctotal": 42,
            # What is the sprint velocity trend (current velocity / average velocity)?
            # 1.0 = stable, >1 = improving, <1 = declining
            # If not agile, enter 1.0 (neutral)
            # [0.5 to 2.0 typically]
            "veltrend": 1.08,
            # How many technical debt items were introduced this period?
            # [whole number, 0 or more] — lower is better
            "tdadded": 5,
            # How many technical debt items were resolved this period?
            # [whole number, 0 or more]
            "tdresolved": 3,
            # What fraction of integration/system tests passed?
            # Example: 92% passed → 0.92
            # [0.0 to 1.0]
            "intpass": 0.92,
            # What fraction of deployments succeeded without rollback?
            # Example: 95% success → 0.95
            # [0.0 to 1.0]
            "deplsucc": 0.95,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 14 — PROJECT GOVERNANCE & REPORTING
        # "Is this PM running a disciplined, transparent,
        #  well-governed project?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many status reports accurately reflected the actual project state?
            # [whole number, 0 or more]
            "srpacc": 11,
            # How many total status reports were submitted?
            # [whole number, 1 or more]
            "srptotal": 12,
            # How many issues were resolved?
            # [whole number, 0 or more]
            "issres": 14,
            # How many issues were resolved within the SLA?
            # [whole number, 0 or more]
            "issressla": 12,
            # How many required project documents were completed?
            # [whole number, 0 or more]
            "docscomp": 18,
            # How many total required project documents were there?
            # [whole number, 1 or more]
            "docstotal": 20,
            # How many retrospectives or lessons-learned sessions were held?
            # [whole number, 0 or more]
            "retroheld": 4,
            # What is the actual resource utilisation rate?
            # 1.0 = fully utilised, 0.88 = 88% utilised
            # [0.0 to 1.0]
            "resutil": 0.88,
            # What is the planned/ideal resource utilisation rate?
            # [0.0 to 1.0]
            "resutiltgt": 1.0,
            # How many steering/governance meetings were held?
            # [whole number, 0 or more]
            "govheld": 10,
            # How many governance meetings were required?
            # [whole number, 1 or more]
            "govreq": 12,
            # What is the project audit compliance score?
            # 0 = non-compliant, 1 = fully compliant
            # [0.0 to 1.0]
            "auditcomp": 0.90,
            # What fraction of the team is using mandated PM tools correctly?
            # Example: 85% adoption → 0.85
            # [0.0 to 1.0]
            "tooladopt": 0.85,
            # What is the average number of days to respond to escalations?
            # [any positive number, can be decimal] — lower is better
            "escrespdays": 1.5,
            # What is the SLA for escalation response (in days)?
            # [any positive number, can be decimal]
            "escsla": 2.0,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 15 — STAKEHOLDER & TEAM MANAGEMENT
        # "Is this PM keeping stakeholders informed, the team
        #  happy, and vendors coordinated?"
        # ═══════════════════════════════════════════════════════════════
        {
            # What is the team satisfaction score with this PM (out of 5)?
            # [0.0 to 5.0]
            "tmsat": 4.0,
            # Historical team satisfaction scores (list of scores out of 5).
            # Used to assess trend/consistency.
            # [list of numbers, each 0.0 to 5.0]
            "tmsathist": [3.6, 3.8, 4.0, 3.7, 3.9, 4.1],
            # What fraction of stakeholder communications were sent on time?
            # Example: 92% on time → 0.92
            # [0.0 to 1.0]
            "stkcomm": 0.92,
            # How many issues arose from poor vendor coordination?
            # [whole number, 0 or more] — lower is better
            "vendiss": 2,
            # How many total vendor coordination touchpoints were there?
            # [whole number, 1 or more]
            "vendtotal": 15,
            # How many client escalations were made about PM performance?
            # [whole number, 0 or more] — lower is better
            "cliesc": 1,
            # What fraction of requirements were documented with clear acceptance criteria?
            # Example: 85% clear → 0.85
            # [0.0 to 1.0]
            "reqclar": 0.85,
            # How many cross-team dependencies were managed successfully?
            # [whole number, 0 or more]
            "ctdepmgd": 8,
            # How many total cross-team dependencies existed?
            # [whole number, 1 or more]
            "ctdeptot": 10,
            # How many knowledge transfer sessions were held for team onboarding?
            # [whole number, 0 or more]
            "ktsess": 3,
            # What is the quality rating of the handoff to BAU/operations?
            # 0 = terrible handoff, 1 = seamless transition
            # [0.0 to 1.0]
            "handoffq": 0.88,
            # How many process improvement suggestions were implemented?
            # [whole number, 0 or more]
            "innovsug": 2,
        },
    ]
elif role == "hr":
    hr_data= [
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 13 — TALENT ACQUISITION & WORKFORCE PLANNING
        # "Is this HR professional hiring the right people,
        #  fast enough, and planning workforce needs effectively?"
        # ═══════════════════════════════════════════════════════════════
        {
            # What is the average number of days to fill a role?
            # [any positive number, can be decimal] — lower is better
            "ttf": 32,
            # What is the benchmark/target days to fill?
            # [any positive number, can be decimal]
            "ttfbench": 28,
            # How many offers were accepted?
            # [whole number, 0 or more]
            "offacc": 18,
            # How many offers were extended?
            # [whole number, 1 or more]
            "offmade": 20,
            # What is the average 90-day performance rating of new hires (out of 5)?
            # [0.0 to 5.0]
            "qoh90": 3.9,
            # Historical quality-of-hire 90-day scores (list of scores out of 5).
            # Used to assess trend/consistency.
            # [list of numbers, each 0.0 to 5.0]
            "qoh90hist": [3.5, 3.7, 3.8, 4.0, 3.6, 3.9, 4.1, 3.8],
            # How many voluntary departures occurred in scope?
            # [whole number, 0 or more] — lower is better
            "volleavers": 5,
            # What is the total headcount under this HR professional's scope?
            # [whole number, 1 or more]
            "hcscope": 120,
            # How many internal transfers/promotions were facilitated?
            # [whole number, 0 or more]
            "intxfer": 4,
            # What is the diversity metric score (org-defined)?
            # 0 = no diversity, 1 = fully diverse per org standards
            # [0.0 to 1.0]
            "divscore": 0.72,
            # What fraction of hires came from proactive sourcing vs. reactive?
            # Example: 65% proactive → 0.65
            # [0.0 to 1.0]
            "srceff": 0.65,
            # What is the candidate experience survey score (out of 5)?
            # [0.0 to 5.0]
            "candexp": 4.2,
            # Historical candidate experience scores (list of scores out of 5).
            # [list of numbers, each 0.0 to 5.0]
            "candexphist": [3.8, 4.0, 4.1, 3.9, 4.2, 4.0],
            # What is the hiring manager satisfaction score (out of 5)?
            # [0.0 to 5.0]
            "hmsat": 4.0,
            # Historical hiring manager satisfaction scores (list of scores out of 5).
            # [list of numbers, each 0.0 to 5.0]
            "hmsathist": [3.6, 3.8, 3.9, 4.0, 3.7, 4.1],
            # How many high-performer voluntary leavers (regrettable attrition)?
            # [whole number, 0 or more] — lower is better
            "regattrn": 2,
            # Total voluntary leavers (for regrettable ratio calculation).
            # [whole number, 1 or more]
            "totvollv": 5,
            # What fraction of the headcount plan vs. actual needs were met?
            # Example: 85% accuracy → 0.85
            # [0.0 to 1.0]
            "wfplacc": 0.85,
            # What fraction of offers were at or above market median?
            # Example: 90% competitive → 0.90
            # [0.0 to 1.0]
            "offcomp": 0.90,
            # How many interviews were conducted per offer (efficiency)?
            # Example: 3.5 interviews per offer
            # [any positive number, can be decimal] — lower is more efficient
            "i2o": 3.5,
            # How many hires left within 6 months (early turnover)?
            # [whole number, 0 or more] — lower is better
            "earlyturn": 1,
            # How many total hires were made this period?
            # [whole number, 1 or more]
            "tothires": 18,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 14 — HR OPERATIONS & COMPLIANCE
        # "Is this HR professional running operations smoothly,
        #  staying compliant, and keeping employees engaged?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many ER/employee cases were resolved within SLA?
            # [whole number, 0 or more]
            "ercasessla": 22,
            # How many total ER/employee cases were there?
            # [whole number, 1 or more]
            "ercasestot": 25,
            # What fraction of the organisation is in HR policy compliance?
            # Example: 93% compliant → 0.93
            # [0.0 to 1.0]
            "polcomp": 0.93,
            # How many L&D programmes were delivered?
            # [whole number, 0 or more]
            "trndel": 12,
            # How many L&D programmes were planned?
            # [whole number, 1 or more]
            "trnplan": 14,
            # What is the current employee engagement score?
            # [0.0 to 5.0, or whatever scale is used]
            "engcurr": 3.8,
            # What was the prior period engagement score?
            # [0.0 to 5.0]
            "engprior": 3.6,
            # How many formal grievances per 100 employees?
            # [any number, 0 or more] — lower is better
            "formgriev": 3,
            # What is the manager satisfaction score with the HRBP (out of 5)?
            # [0.0 to 5.0]
            "hrbpsat": 4.1,
            # Historical HRBP satisfaction scores (list of scores out of 5).
            # [list of numbers, each 0.0 to 5.0]
            "hrbpsathist": [3.5, 3.8, 3.9, 4.0, 3.7, 4.2, 4.1],
            # What fraction of payroll runs were processed without error?
            # Example: 99.5% accurate → 0.995
            # [0.0 to 1.0]
            "payacc": 0.995,
            # What fraction of eligible employees enrolled in benefits?
            # Example: 92% enrolled → 0.92
            # [0.0 to 1.0]
            "benenroll": 0.92,
            # What fraction of mandatory compliance training was completed?
            # Example: 96% completed → 0.96
            # [0.0 to 1.0]
            "comptrn": 0.96,
            # How many HR audit findings were there?
            # [whole number, 0 or more] — lower is better
            "auditfind": 1,
            # What is the HRIS data accuracy rate?
            # Example: 97% accurate → 0.97
            # [0.0 to 1.0]
            "hrisacc": 0.97,
            # What fraction of new hires completed full onboarding?
            # Example: 88% completed → 0.88
            # [0.0 to 1.0]
            "onbcomp": 0.88,
            # What fraction of departing employees had exit interviews?
            # Example: 75% completed → 0.75
            # [0.0 to 1.0]
            "exitint": 0.75,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 15 — PEOPLE EXPERIENCE & STRATEGIC HR
        # "Is this HR professional driving culture, succession,
        #  and strategic people initiatives?"
        # ═══════════════════════════════════════════════════════════════
        {
            # What is the Employee Net Promoter Score?
            # [-100 to 100]
            "enps": 35,
            # What is the benchmark eNPS?
            # [-100 to 100]
            "enpsbench": 30,
            # What fraction of key roles have identified successors?
            # Example: 60% covered → 0.60
            # [0.0 to 1.0]
            "succcov": 0.60,
            # How many DEI initiatives were completed?
            # [whole number, 0 or more]
            "deicomp": 3,
            # How many DEI initiatives were planned?
            # [whole number, 1 or more]
            "deiplan": 4,
            # How many coaching sessions were delivered to managers?
            # [whole number, 0 or more]
            "mgrcoach": 8,
            # What is the average hours to respond to employee queries?
            # [any positive number, can be decimal] — lower is better
            "hrresp": 6,
            # What is the SLA for HR response time (in hours)?
            # [any positive number, can be decimal]
            "hrrespsla": 8,
            # What fraction of employees participated in culture/pulse surveys?
            # Example: 82% participation → 0.82
            # [0.0 to 1.0]
            "cultpart": 0.82,
            # What fraction of employees used wellness programmes?
            # Example: 45% uptake → 0.45
            # [0.0 to 1.0]
            "welluptk": 0.45,
            # What is the internal mobility ratio (internal fills / total fills)?
            # Example: 12% internal → 0.12
            # [0.0 to 1.0]
            "intmob": 0.12,
            # What fraction of employees had a completed talent review?
            # Example: 90% reviewed → 0.90
            # [0.0 to 1.0]
            "talrev": 0.90,
            # What fraction of HR-led PIPs resulted in improvement?
            # Example: 40% improved → 0.40
            # [0.0 to 1.0]
            "pipsucc": 0.40,
            # How many total PIPs were managed by HR?
            # [whole number, 1 or more]
            "piptot": 5,
            # What fraction of managers are using formal recognition tools?
            # Example: 65% usage → 0.65
            # [0.0 to 1.0]
            "recoguse": 0.65,
        },
    ]
elif role == "accounts":
    accounts_data= [
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 13 — FINANCIAL ACCURACY & REPORTING
        # "Is this accountant producing accurate financials,
        #  closing on time, and keeping forecasts tight?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many financial restatements were caused?
            # [whole number, 0 or more] — lower is better (0 is ideal)
            "restate": 0,
            # How many month/year-end closes were completed on time?
            # [whole number, 0 or more]
            "closesot": 11,
            # How many total close cycles were there?
            # [whole number, 1 or more]
            "closestot": 12,
            # How many reconciliations were completed without error?
            # [whole number, 0 or more]
            "cleanrec": 145,
            # How many total reconciliations were performed?
            # [whole number, 1 or more]
            "totrec": 150,
            # How many budget vs. actual variances were explained within SLA?
            # [whole number, 0 or more]
            "varexpsla": 28,
            # How many total variances needed explanation?
            # [whole number, 1 or more]
            "vartotal": 32,
            # How many material audit findings were attributable to this person?
            # [whole number, 0 or more] — lower is better (0 is ideal)
            "mataudit": 0,
            # What was the financial forecast ($)?
            # [any number, 0 or more]
            "finfc": 4800000,
            # What was the actual financial result ($)?
            # [any number, 0 or more]
            "finact": 5000000,
            # How many regulatory filings were compliant?
            # [whole number, 0 or more]
            "regcomp": 8,
            # How many total regulatory filings were there?
            # [whole number, 1 or more]
            "regtotal": 8,
            # What fraction of journal entries were correct on first submission?
            # Example: 98% correct → 0.98
            # [0.0 to 1.0]
            "jeacc": 0.98,
            # What is the intercompany reconciliation accuracy?
            # Example: 95% accurate → 0.95
            # [0.0 to 1.0]
            "icrecacc": 0.95,
            # What fraction of financial reports were delivered on time?
            # Example: 92% on time → 0.92
            # [0.0 to 1.0]
            "rptot": 0.92,
            # What is the accrual accuracy (accrual estimates vs. actuals)?
            # Example: 94% accurate → 0.94
            # [0.0 to 1.0]
            "accracc": 0.94,
            # What is the tax provision accuracy?
            # Example: 96% accurate → 0.96
            # [0.0 to 1.0]
            "taxacc": 0.96,
            # What fraction of balance sheet accounts were reviewed per schedule?
            # Example: 90% reviewed → 0.90
            # [0.0 to 1.0]
            "bsrev": 0.90,
            # What fraction of month-end checklist items were completed?
            # Example: 95% completed → 0.95
            # [0.0 to 1.0]
            "mechk": 0.95,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 14 — CONTROLS, RISK & COMPLIANCE
        # "Is this accountant maintaining strong controls,
        #  managing risk, and staying compliant?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many SOX / internal control exceptions occurred?
            # [whole number, 0 or more] — lower is better (0 is ideal)
            "soxexc": 0,
            # How many payables were processed within payment terms?
            # [whole number, 0 or more]
            "apinterm": 420,
            # How many total payables were there?
            # [whole number, 1 or more]
            "aptotal": 450,
            # How many receivables were collected within terms?
            # [whole number, 0 or more]
            "arinterm": 380,
            # How many total receivables were there?
            # [whole number, 1 or more]
            "artotal": 400,
            # How many duplicate or erroneous payments were made?
            # [whole number, 0 or more] — lower is better
            "duppay": 1,
            # What is the finance policy audit adherence score?
            # 0 = non-adherent, 1 = fully adherent
            # [0.0 to 1.0]
            "finpoladh": 0.92,
            # How many data integrity checks passed?
            # [whole number, 0 or more]
            "dichkpass": 48,
            # How many total data integrity checks were performed?
            # [whole number, 1 or more]
            "dichktot": 50,
            # What is the segregation of duties compliance rate?
            # Example: 98% compliant → 0.98
            # [0.0 to 1.0]
            "sodcomp": 0.98,
            # What fraction of system access reviews were completed?
            # Example: 100% completed → 1.0
            # [0.0 to 1.0]
            "accrev": 1.0,
            # What fraction of vendor master changes had proper authorisation?
            # Example: 97% authorised → 0.97
            # [0.0 to 1.0]
            "vmauth": 0.97,
            # What fraction of bank reconciliations were completed within the deadline?
            # Example: 95% on time → 0.95
            # [0.0 to 1.0]
            "bnkrec": 0.95,
            # What fraction of write-offs were properly authorised?
            # Example: 100% authorised → 1.0
            # [0.0 to 1.0]
            "woauth": 1.0,
            # How many potential fraud indicators were flagged proactively?
            # [whole number, 0 or more]
            "fraudflag": 3,
            # How many actual fraud indicators existed (for detection rate)?
            # [whole number, 1 or more]
            "fraudact": 3,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 15 — STAKEHOLDER SERVICE & PROCESS IMPROVEMENT
        # "Is this accountant serving internal customers well
        #  and driving continuous improvement?"
        # ═══════════════════════════════════════════════════════════════
        {
            # What is the internal customer (business partner) satisfaction score (out of 5)?
            # [0.0 to 5.0]
            "bpsat": 4.0,
            # Historical business partner satisfaction scores (list of scores out of 5).
            # [list of numbers, each 0.0 to 5.0]
            "bpsathist": [3.5, 3.8, 4.0, 3.7, 3.9, 4.1],
            # What is the average hours to resolve finance queries?
            # [any positive number, can be decimal] — lower is better
            "qreshrs": 18,
            # What is the SLA for query resolution (in hours)?
            # [any positive number, can be decimal]
            "qressla": 24,
            # How many process improvements were implemented?
            # [whole number, 0 or more]
            "procimp": 3,
            # How many manual processes were automated?
            # [whole number, 0 or more]
            "autoinit": 2,
            # How much in cost savings was identified ($)?
            # [any number, 0 or more]
            "costsav": 45000,
            # What is the cost savings target ($)?
            # [any number, 1 or more]
            "costsavtgt": 50000,
            # How many finance training sessions were delivered to non-finance staff?
            # [whole number, 0 or more]
            "trnsess": 2,
            # What fraction of stakeholders are using self-serve dashboards?
            # Example: 75% adoption → 0.75
            # [0.0 to 1.0]
            "dashadopt": 0.75,
            # What is the finance system availability/uptime maintained?
            # Example: 99% uptime → 0.99
            # [0.0 to 1.0]
            "sysup": 0.99,
            # What fraction of SOPs/process documents are current?
            # Example: 85% current → 0.85
            # [0.0 to 1.0]
            "doccurr": 0.85,
        },
    ]
elif role == "teammanager":
    team_manager_data= [
    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 13 — TEAM OUTCOMES & PEOPLE DEVELOPMENT
    # "Is this manager growing their people, retaining
    #  talent, and lifting team performance?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What is the average employee score of direct reports?
        # 0 = lowest performance, 1 = highest performance
        # [0.0 to 1.0]
        "tmavg": 0.74,
        # What was the prior period team average score?
        # [0.0 to 1.0]
        "tmavgprior": 0.70,
        # How many voluntary departures occurred from the team?
        # [whole number, 0 or more] — lower is better
        "voldep": 2,
        # What is the total team size?
        # [whole number, 1 or more]
        "tmsize": 15,
        # How many team members were promoted?
        # [whole number, 0 or more]
        "tmpromo": 2,
        # How many key roles have identified successors?
        # [whole number, 0 or more]
        "keysucc": 3,
        # How many total key roles are there?
        # [whole number, 1 or more]
        "keytot": 5,
        # How many PIPs led to improvement?
        # [whole number, 0 or more]
        "pipimp": 1,
        # How many total PIPs were initiated?
        # [whole number, 1 or more]
        "piptot": 2,
        # How many 1:1 meetings were actually held?
        # [whole number, 0 or more]
        "o2oheld": 46,
        # How many 1:1 meetings were scheduled?
        # [whole number, 1 or more]
        "o2osched": 48,
        # What is the average training hours per team member?
        # [any positive number, can be decimal]
        "trnhrs": 12,
        # What is the target training hours per team member?
        # [any positive number, can be decimal]
        "trnhrstgt": 16,
        # What fraction of high performers were retained?
        # Example: 100% retained → 1.0
        # [0.0 to 1.0]
        "hpret": 1.0,
        # What is the team engagement survey score (out of 5)?
        # [0.0 to 5.0]
        "tmeng": 4.0,
        # Historical team engagement scores (list of scores out of 5).
        # [list of numbers, each 0.0 to 5.0]
        "tmenghist": [3.6, 3.8, 3.9, 4.0, 3.7, 4.1],
        # How many team members have documented career plans?
        # [whole number, 0 or more]
        "carplan": 12,
        # What is the target for career plans (usually = team size)?
        # [whole number, 1 or more]
        "carplantgt": 15,
        # What is the average days for a new hire to reach productivity?
        # [any positive number, can be decimal] — lower is better
        "rampdays": 35,
        # What is the benchmark ramp days?
        # [any positive number, can be decimal]
        "ramptgt": 30,
        # What fraction of performance reviews were completed on time?
        # Example: 100% completed → 1.0
        # [0.0 to 1.0]
        "revcomp": 1.0,
        # How many skills gaps were identified and addressed?
        # [whole number, 0 or more]
        "skgclosed": 4,
        # How many total skills gaps were identified?
        # [whole number, 1 or more]
        "skgtot": 6,
    },
    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 14 — OPERATIONAL MANAGEMENT
    # "Is this manager running an efficient, well-resourced,
    #  on-budget operation?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What is the actual productive headcount?
        # [whole number, 0 or more]
        "prodhc": 14,
        # What is the allocated headcount?
        # [whole number, 1 or more]
        "allochc": 15,
        # What is the actual team budget spend ($)?
        # [any number, 0 or more]
        "tmbudact": 310000,
        # What is the allocated team budget ($)?
        # [any number, 1 or more]
        "tmbudalloc": 300000,
        # How many cross-functional projects were delivered on time?
        # [whole number, 0 or more]
        "cfdelot": 5,
        # How many total cross-functional projects were there?
        # [whole number, 1 or more]
        "cfdeltot": 6,
        # What is the upward feedback score from direct reports (anonymous, out of 5)?
        # [0.0 to 5.0]
        "upfb": 4.1,
        # Historical upward feedback scores (list of scores out of 5).
        # [list of numbers, each 0.0 to 5.0]
        "upfbhist": [3.5, 3.8, 3.9, 4.0, 4.1, 3.7, 4.2],
        # What is the average days for team-level decisions?
        # [any positive number, can be decimal] — lower is better
        "decdays": 2.5,
        # What is the benchmark days for team-level decisions?
        # [any positive number, can be decimal]
        "decdaysbench": 2.0,
        # What fraction of total work hours is spent in meetings?
        # Example: 18% meeting time → 0.18
        # [0.0 to 1.0] — there's a sweet spot; too high or too low is bad
        "mtghrs": 0.18,
        # What is the team SLA compliance rate?
        # Example: 90% met → 0.90
        # [0.0 to 1.0]
        "slamet": 0.90,
        # What is the capacity planning accuracy (resource forecast vs. actual needs)?
        # Example: 85% accurate → 0.85
        # [0.0 to 1.0]
        "capacc": 0.85,
        # What fraction of the team follows standard processes?
        # Example: 92% compliance → 0.92
        # [0.0 to 1.0]
        "proccomp": 0.92,
        # What is the average hours to respond to team incidents?
        # [any positive number, can be decimal] — lower is better
        "incresp": 2.0,
        # What is the SLA for incident response (in hours)?
        # [any positive number, can be decimal]
        "increspsla": 4.0,
        # What fraction of delegated tasks were completed successfully?
        # Example: 80% successful → 0.80
        # [0.0 to 1.0]
        "delgeff": 0.80,
    },
    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 15 — CULTURE, INCLUSION & STRATEGIC LEADERSHIP
    # "Is this manager building an inclusive, innovative,
    #  psychologically safe culture and contributing
    #  to org-level strategy?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What is the team diversity/inclusion metric?
        # 0 = no diversity, 1 = fully diverse per org standards
        # [0.0 to 1.0]
        "tmdei": 0.78,
        # What is the psychological safety survey score (out of 5)?
        # [0.0 to 5.0]
        "psysafe": 4.0,
        # Historical psychological safety scores (list of scores out of 5).
        # [list of numbers, each 0.0 to 5.0]
        "psysafehist": [3.5, 3.8, 3.9, 4.0, 3.7],
        # How many recognition moments were given per quarter?
        # [whole number, 0 or more]
        "recogfreq": 12,
        # What is the target recognitions per quarter?
        # [whole number, 1 or more]
        "recogtgt": 15,
        # What fraction of intra-team conflicts were resolved without escalation?
        # Example: 85% resolved internally → 0.85
        # [0.0 to 1.0]
        "confres": 0.85,
        # What fraction of the team adopted new processes/tools within the target period?
        # Example: 80% adoption → 0.80
        # [0.0 to 1.0]
        "chgadopt": 0.80,
        # How many org-level strategic initiatives did this manager contribute to?
        # [whole number, 0 or more]
        "stratinit": 3,
        # What is the target for strategic initiatives?
        # [whole number, 1 or more]
        "stratinittgt": 4,
        # How many ideas were generated by the team and submitted?
        # [whole number, 0 or more]
        "innovideas": 5,
        # What is the feedback loop score (manager acts on upward feedback, from 360 assessment)?
        # 0 = ignores all feedback, 1 = consistently acts on feedback
        # [0.0 to 1.0]
        "fbloop": 0.82,
        # How evenly is work distributed across the team?
        # 0 = extremely uneven, 1 = perfectly balanced
        # [0.0 to 1.0]
        "wleq": 0.88,
    },
]
elif role == "vendor_manager":
    vendor_manager_data=[
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 13 — VENDOR PERFORMANCE & VALUE
        # "Is this vendor manager getting maximum performance
        #  and value from the supply base?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How many SLAs were met by managed vendors?
            # [whole number, 0 or more]
            "slasmet": 85,
            # How many total SLAs were there?
            # [whole number, 1 or more]
            "slastot": 95,
            # What was the actual vendor spend ($)?
            # [any number, 0 or more]
            "actvendspend": 520000,
            # What was the contracted vendor cost ($)?
            # [any number, 1 or more]
            "contrvendcost": 500000,
            # How many contract renewals were on favourable terms?
            # [whole number, 0 or more]
            "renfav": 8,
            # How many total contract renewals were there?
            # [whole number, 1 or more]
            "rentot": 10,
            # How many vendor-caused incidents occurred?
            # [whole number, 0 or more] — lower is better
            "vendinc": 4,
            # How much in cost savings was delivered ($)?
            # [any number, 0 or more]
            "savdel": 42000,
            # What is the cost savings target ($)?
            # [any number, 1 or more]
            "savtgt": 50000,
            # What fraction of spend went to diverse/SME suppliers?
            # Example: 18% → 0.18
            # [0.0 to 1.0]
            "divspend": 0.18,
            # What is the target fraction of spend with diverse/SME suppliers?
            # [0.0 to 1.0]
            "divspendtgt": 0.20,
            # How many critical categories have a backup vendor identified?
            # [whole number, 0 or more]
            "critcatcov": 7,
            # How many total critical categories are there?
            # [whole number, 1 or more]
            "critcattot": 8,
            # What is the average quality rating of managed vendors?
            # 0 = terrible quality, 1 = exceptional quality
            # [0.0 to 1.0]
            "vendqual": 0.82,
            # What fraction of vendor deliveries were on time?
            # Example: 88% on time → 0.88
            # [0.0 to 1.0]
            "venddelot": 0.88,
            # What is the total contract value under management ($)?
            # [any number, 0 or more]
            "contrvalmgd": 2500000,
            # How many joint innovation projects were undertaken with vendors?
            # [whole number, 0 or more]
            "vendinnov": 2,
            # How many vendor consolidations were completed?
            # [whole number, 0 or more]
            "vendcons": 3,
            # What is the target for vendor consolidations?
            # [whole number, 1 or more]
            "vendconstgt": 4,
            # How many days of payment terms were extended vs. prior period?
            # [whole number or decimal, 0 or more]
            "paytermopt": 5,
            # What is the target days of payment term extension?
            # [whole number or decimal, 1 or more]
            "paytermtgt": 7,
            # What is the total cost of ownership reduction achieved?
            # Example: 5% reduction → 0.05
            # [0.0 to 1.0]
            "tcored": 0.05,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 14 — PROCUREMENT PROCESS & COMPLIANCE
        # "Is this vendor manager following procurement policy,
        #  managing risk, and maintaining spend discipline?"
        # ═══════════════════════════════════════════════════════════════
        {
            # How much spend is under formal contract ($)?
            # [any number, 0 or more]
            "spundcontr": 480000,
            # What is the total vendor spend ($)?
            # [any number, 1 or more]
            "totspend": 520000,
            # How many RFP/tenders followed procurement policy?
            # [whole number, 0 or more]
            "rfpcomp": 12,
            # How many total RFP/tenders were conducted?
            # [whole number, 1 or more]
            "rfptot": 14,
            # How many vendors have completed due diligence?
            # [whole number, 0 or more]
            "venddd": 28,
            # How many total active vendors are there?
            # [whole number, 1 or more]
            "vendact": 32,
            # How many purchases had a proper PO?
            # [whole number, 0 or more]
            "purchpo": 195,
            # How many total purchases were made?
            # [whole number, 1 or more]
            "purchtot": 200,
            # What is the average days from invoice receipt to approval?
            # [any positive number, can be decimal] — lower is better
            "invapprdays": 4,
            # What is the target days for invoice approval?
            # [any positive number, can be decimal]
            "invapprtgt": 5,
            # How many vendors have an assessed risk profile?
            # [whole number, 0 or more]
            "vendriskass": 26,
            # How many total vendors need risk assessment?
            # [whole number, 1 or more]
            "vendrisktot": 32,
            # What fraction of spend is outside approved channels (maverick spend)?
            # Example: 8% maverick → 0.08
            # [0.0 to 1.0] — lower is better
            "mavspend": 0.08,
            # What fraction of contracts were renewed/replaced before expiry?
            # Example: 95% managed → 0.95
            # [0.0 to 1.0]
            "contrexpmgd": 0.95,
            # What is the average days to onboard a new supplier?
            # [any positive number, can be decimal] — lower is better
            "supponbdays": 12,
            # What is the target days for supplier onboarding?
            # [any positive number, can be decimal]
            "supponbtgt": 10,
            # What fraction of vendors meet sustainability criteria?
            # Example: 80% compliant → 0.80
            # [0.0 to 1.0]
            "sustcomp": 0.80,
            # What fraction of vendor insurance certificates are current?
            # Example: 92% current → 0.92
            # [0.0 to 1.0]
            "inscert": 0.92,
        },
        # ═══════════════════════════════════════════════════════════════
        # PARAMETER 15 — RELATIONSHIP & STRATEGIC SOURCING
        # "Is this vendor manager building strong vendor
        #  partnerships and driving strategic sourcing?"
        # ═══════════════════════════════════════════════════════════════
        {
            # What is the vendor satisfaction score with this manager (out of 5)?
            # [0.0 to 5.0]
            "vendsat": 4.0,
            # Historical vendor satisfaction scores (list of scores out of 5).
            # [list of numbers, each 0.0 to 5.0]
            "vendsathist": [3.5, 3.7, 3.9, 4.0, 3.8, 4.1],
            # What fraction of strategic vendors had a QBR/annual review completed?
            # Example: 85% completed → 0.85
            # [0.0 to 1.0]
            "stratrev": 0.85,
            # How many vendor escalations were resolved?
            # [whole number, 0 or more]
            "vendescres": 6,
            # How many total vendor escalations were there?
            # [whole number, 1 or more]
            "vendesctot": 7,
            # How many market/category analysis reports were delivered?
            # [whole number, 0 or more]
            "mktintel": 3,
            # What is the target for market intel reports?
            # [whole number, 1 or more]
            "mktinteltgt": 4,
            # What is the internal stakeholder satisfaction score for procurement (out of 5)?
            # [0.0 to 5.0]
            "stksat": 4.1,
            # Historical internal stakeholder satisfaction scores (list of scores out of 5).
            # [list of numbers, each 0.0 to 5.0]
            "stksathist": [3.6, 3.8, 4.0, 3.9, 4.1],
            # How many vendors have a formal development/improvement plan?
            # [whole number, 0 or more]
            "venddevplan": 4,
            # What is the target for vendor development plans?
            # [whole number, 1 or more]
            "venddevtgt": 5,
            # What fraction of spend categories have a documented strategy?
            # Example: 75% covered → 0.75
            # [0.0 to 1.0]
            "catstrat": 0.75,
            # How many cross-functional sourcing initiatives were undertaken?
            # [whole number, 0 or more]
            "cfsrcproj": 3,
            # What is the legal team's rating of collaboration with this vendor manager?
            # 0 = poor collaboration, 1 = excellent collaboration
            # [0.0 to 1.0]
            "legalcollab": 0.88,
            # What fraction of projects had procurement engaged before the RFP stage?
            # Example: 70% early engagement → 0.70
            # [0.0 to 1.0]
            "earlyeng": 0.70,
        },
    ]

In [18]:
if role=="sales":
    d=sales_data[0]
    pipeline = min(d["pv"] / d["pvtarget"], 1.2) / 1.2
    conv_rate = bci(d["lconv"], d["lworked"])
    deal_count = ln_norm(d["dclosed"], 20)
    deal_size = min(d["avgdeal"] / d["avgdealbench"], 1.3) / 1.3
    cycle_penalty = edp(max(0, d["scdays"] - d["scdaysbench"]), 0.05)
    upsell = min(d["uprev"] / max(d["baserev"], 1), 1.0)
    churn = max(0, 1 - 0.25 * d["achurn"])
    forecast_acc = 1 - min(abs(d["fcsub"] - d["fcactual"])
                           / max(d["fcactual"], 1), 1)
    quota_attain = min(d["quotaach"] / max(d["quota"], 1), 1.3) / 1.3
    new_logos = min(d["newlogos"] / max(d["newlogotgt"], 1), 1.2) / 1.2
    wpipe_acc = d["wpacc"]
    demo_to_prop = d["d2p"]
    prop_win = d["pwin"]
    discount_eff = min(d["discbench"] / max(d["discavg"], 0.1), 1.3) / 1.3
    rev_conc = edp(d["revconc"], 2.0)
    multi_year = bci(d["mydeals"], d["totaldeals"])
    X13 = hrc(
        values=[pipeline, conv_rate, deal_count, deal_size, cycle_penalty,
                upsell, churn, forecast_acc, quota_attain, new_logos,
                wpipe_acc, demo_to_prop, prop_win, discount_eff, rev_conc, multi_year],
        weights=[0.10, 0.09, 0.06, 0.07, 0.06,
                 0.06, 0.06, 0.07, 0.10, 0.06,
                 0.05, 0.04, 0.05, 0.05, 0.04, 0.04]
    )

    d=sales_data[1]
    crm = bci(d["crmcorr"], d["crmtotal"])
    activity = min(d["callsmade"] / max(d["callstgt"], 1), 1.2) / 1.2
    prop_quality = d["propqual"] / 5.0
    discovery = bci(d["discdoc"], d["disctotal"])
    deals_lost_err = max(0, 1 - 0.20 * d["dlostrerr"])
    stage_updates = d["stageupd"]
    followup = d["fucomp"]
    handoff = d["handoff"]
    comp_intel = ln_norm(d["cintel"], 8)
    territory = d["tpadh"]
    quote_acc = d["qtacc"]
    X14 = hrc(
        values=[crm, activity, prop_quality, discovery, deals_lost_err,
                stage_updates, followup, handoff, comp_intel, territory, quote_acc],
        weights=[0.12, 0.10, 0.10, 0.10, 0.08,
                 0.08, 0.10, 0.08, 0.06, 0.08, 0.10]
    )

    d=sales_data[2]
    exec_contacts = min(d["execcon"] / max(d["execcontgt"], 1), 1.0)
    referrals = ln_norm(d["refgen"], 6)
    nps = min(max(d["nps"], 0) / 100.0, 1.0)
    engagement = d["custeng"]
    strategic_cov = d["stracov"]
    partner_deals = ln_norm(d["partdeals"], 5)
    event_conv = bci(d["evleadconv"], d["evleadtot"])
    thought_lead = ln_norm(d["tlcontrib"], 5)
    health_updates = d["chupd"]
    expansion = min(d["expplan"] / max(d["expplantgt"], 1), 1.0)
    X15 = hrc(
        values=[exec_contacts, referrals, nps, engagement, strategic_cov,
                partner_deals, event_conv, thought_lead, health_updates, expansion],
        weights=[0.10, 0.10, 0.12, 0.12, 0.12,
                 0.08, 0.10, 0.06, 0.10, 0.10]
    )
    
elif role=="projects":
    d=projects_data[0]
    scope = bci(d["delscope"], d["deltotal"])
    budget_var = edp(abs(d["actspend"] - d["budget"]) / max(d["budget"], 1), 4.0)
    spi = min(d["spi"], 1.2) / 1.2
    cpi = min(d["cpi"], 1.2) / 1.2
    milestones = bci(d["mshit"], d["mstotal"])
    risk_quality = d["riskearly"] / max(d["risktotal"], 1)
    stakeholder_sat = d["stksat"] / 5.0
    change_req = edp(d["chgreq"] / max(d["origscope"], 1), 2.0)
    dependency = 1 - min(d["blkdays"] / max(d["projdays"], 1), 1)
    defect_rate = edp(d["defects"] / max(d["deldel"], 1), 1.5)
    rework = max(0, 1 - 0.20 * d["rework"])
    acceptance = bci(d["accmet"], d["acctotal"])
    velocity = gaussian_penalty(d["veltrend"], 1.0, 0.20)
    tech_debt_net = max(0, 1 - 0.15 * max(0, d["tdadded"] - d["tdresolved"]))
    integration_test = d["intpass"]
    deploy_success = d["deplsucc"]
    X13 = hrc(
        values=[scope, budget_var, spi, cpi, milestones, risk_quality,
                stakeholder_sat, change_req, dependency, defect_rate,
                rework, acceptance, velocity, tech_debt_net,
                integration_test, deploy_success],
        weights=[0.08, 0.08, 0.08, 0.07, 0.08, 0.07,
                 0.07, 0.06, 0.05, 0.06,
                 0.05, 0.06, 0.05, 0.04,
                 0.05, 0.05]
    )

    d=projects_data[1]
    status_acc = bci(d["srpacc"], d["srptotal"])
    issue_res = bci(d["issressla"], d["issres"])
    docs = d["docscomp"] / max(d["docstotal"], 1)
    retros = ln_norm(d["retroheld"], 6)
    resource_util = gaussian_penalty(d["resutil"], d["resutiltgt"], 0.15)
    governance = bci(d["govheld"], d["govreq"])
    audit = d["auditcomp"]
    tool_adoption = d["tooladopt"]
    esc_response = edp(max(0, d["escrespdays"] - d["escsla"]), 0.5)
    X14 = hrc(
        values=[status_acc, issue_res, docs, retros, resource_util,
                governance, audit, tool_adoption, esc_response],
        weights=[0.15, 0.15, 0.10, 0.08, 0.12,
                 0.10, 0.10, 0.10, 0.10]
    )
  
    d=projects_data[2]
    team_sat = d["tmsat"] / 5.0
    comms_ontime = d["stkcomm"]
    vendor_coord = 1 - bci(d["vendiss"], d["vendtotal"])
    client_esc = max(0, 1 - 0.25 * d["cliesc"])
    req_clarity = d["reqclar"]
    dep_managed = bci(d["ctdepmgd"], d["ctdeptot"])
    kt_sessions = ln_norm(d["ktsess"], 6)
    handoff = d["handoffq"]
    innovation = ln_norm(d["innovsug"], 5)
    X15 = hrc(
        values=[team_sat, comms_ontime, vendor_coord, client_esc,
                req_clarity, dep_managed, kt_sessions, handoff, innovation],
        weights=[0.15, 0.12, 0.10, 0.10,
                 0.12, 0.12, 0.08, 0.12, 0.09]
    )

elif role=="hr":
    d=hr_data[0]
    ttf = edp(max(0, d["ttf"] - d["ttfbench"]), 0.03)
    offer_accept = bci(d["offacc"], d["offmade"])
    qoh = d["qoh90"] / 5.0
    attrition = edp(d["volleavers"] / max(d["hcscope"], 1), 5.0)
    internal_mob = ln_norm(d["intxfer"], 10)
    diversity = d["divscore"]
    sourcing_eff = d["srceff"]
    cand_exp = d["candexp"] / 5.0
    hmgr_sat = d["hmsat"] / 5.0
    regrettable = 1 - (d["regattrn"] / max(d["totvollv"], 1))
    wfp_acc = d["wfplacc"]
    offer_comp = d["offcomp"]
    interview_eff = edp(max(0, d["i2o"] - 3.0), 0.3)
    early_turnover = max(0, 1 - 0.30 * d["earlyturn"])
    X13 = hrc(
        values=[ttf, offer_accept, qoh, attrition, internal_mob,
                diversity, sourcing_eff, cand_exp, hmgr_sat, regrettable,
                wfp_acc, offer_comp, interview_eff, early_turnover],
        weights=[0.09, 0.08, 0.10, 0.10, 0.06,
                 0.05, 0.06, 0.06, 0.07, 0.08,
                 0.07, 0.06, 0.05, 0.07]
    )

    d=hr_data[1]
    case_res = bci(d["ercasessla"], d["ercasestot"])
    policy = d["polcomp"]
    training = min(d["trndel"] / max(d["trnplan"], 1), 1.0)
    eng_delta = 0.5 + 0.5 * np.tanh(3 * (d["engcurr"] - d["engprior"]))
    grievance = edp(d["formgriev"], 0.3)
    hrbp_sat = d["hrbpsat"] / 5.0
    payroll = d["payacc"]
    benefits = d["benenroll"]
    compliance_train = d["comptrn"]
    audit = max(0, 1 - 0.30 * d["auditfind"])
    hris = d["hrisacc"]
    onboarding = d["onbcomp"]
    exit_interview = d["exitint"]
    X14 = hrc(
        values=[case_res, policy, training, eng_delta, grievance,
                hrbp_sat, payroll, benefits, compliance_train, audit,
                hris, onboarding, exit_interview],
        weights=[0.10, 0.08, 0.07, 0.09, 0.08,
                 0.08, 0.08, 0.06, 0.08, 0.07,
                 0.07, 0.07, 0.07]
    )

    d=hr_data[2]
    enps = min(max(d["enps"], 0) / 100.0, 1.0)
    succession = d["succcov"]
    dei = min(d["deicomp"] / max(d["deiplan"], 1), 1.0)
    coaching = ln_norm(d["mgrcoach"], 12)
    response = edp(max(0, d["hrresp"] - d["hrrespsla"]), 0.1)
    culture_part = d["cultpart"]
    wellness = d["welluptk"]
    mobility = min(d["intmob"] / 0.20, 1.0)
    talent_review = d["talrev"]
    pip_success = bci(round(d["pipsucc"] * d["piptot"]), d["piptot"])
    recognition = d["recoguse"]
    X15 = hrc(
        values=[enps, succession, dei, coaching, response,
                culture_part, wellness, mobility, talent_review,
                pip_success, recognition],
        weights=[0.10, 0.10, 0.08, 0.08, 0.08,
                 0.08, 0.06, 0.08, 0.10,
                 0.10, 0.14]
    )
    
elif role=="accounts":
    d=accounts_data[0]
    restatement = max(0, 1 - 0.35 * d["restate"])
    close_time = bci(d["closesot"], d["closestot"])
    rec_acc = bci(d["cleanrec"], d["totrec"])
    var_explain = bci(d["varexpsla"], d["vartotal"])
    audit_find = max(0, 1 - 0.40 * d["mataudit"])
    forecast = 1 - min(abs(d["finfc"] - d["finact"])
                       / max(d["finact"], 1), 1)
    compliance = bci(d["regcomp"], d["regtotal"])
    je_accuracy = d["jeacc"]
    interco = d["icrecacc"]
    report_ontime = d["rptot"]
    accrual = d["accracc"]
    tax = d["taxacc"]
    bs_review = d["bsrev"]
    checklist = d["mechk"]
    X13 = hrc(
        values=[restatement, close_time, rec_acc, var_explain, audit_find,
                forecast, compliance, je_accuracy, interco, report_ontime,
                accrual, tax, bs_review, checklist],
        weights=[0.10, 0.08, 0.10, 0.06, 0.10,
                 0.06, 0.08, 0.08, 0.06, 0.06,
                 0.06, 0.06, 0.05, 0.05]
    )

    d=accounts_data[1]
    sox = max(0, 1 - 0.30 * d["soxexc"])
    ap_aging = bci(d["apinterm"], d["aptotal"])
    ar_aging = bci(d["arinterm"], d["artotal"])
    dup_pay = max(0, 1 - 0.50 * d["duppay"])
    policy = d["finpoladh"]
    data_int = bci(d["dichkpass"], d["dichktot"])
    sod = d["sodcomp"]
    access = d["accrev"]
    vendor_master = d["vmauth"]
    bank_rec = d["bnkrec"]
    write_off = d["woauth"]
    fraud_detect = d["fraudflag"] / max(d["fraudact"], 1)
    X14 = hrc(
        values=[sox, ap_aging, ar_aging, dup_pay, policy, data_int,
                sod, access, vendor_master, bank_rec, write_off, fraud_detect],
        weights=[0.12, 0.10, 0.10, 0.10, 0.08, 0.10,
                 0.08, 0.06, 0.06, 0.06, 0.06, 0.08]
    )

    d=accounts_data[2]
    bp_sat = d["bpsat"] / 5.0
    query_res = edp(max(0, d["qreshrs"] - d["qressla"]), 0.1)
    proc_improve = ln_norm(d["procimp"], 6)
    automation = ln_norm(d["autoinit"], 4)
    savings = min(d["costsav"] / max(d["costsavtgt"], 1), 1.2) / 1.2
    training = ln_norm(d["trnsess"], 5)
    dashboards = d["dashadopt"]
    sys_uptime = d["sysup"]
    doc_currency = d["doccurr"]
    X15 = hrc(
        values=[bp_sat, query_res, proc_improve, automation, savings,
                training, dashboards, sys_uptime, doc_currency],
        weights=[0.15, 0.12, 0.12, 0.10, 0.12,
                 0.08, 0.10, 0.10, 0.11]
    )

elif role=="teammanager":
    d=team_manager_data[0]
    team_avg = d["tmavg"]
    team_delta = 0.5 + 0.5 * np.tanh(4 * (d["tmavg"] - d["tmavgprior"]))
    attrition = edp(d["voldep"] / max(d["tmsize"], 1), 6.0)
    promo = ln_norm(d["tmpromo"], round(d["tmsize"] * 0.3))
    succession = d["keysucc"] / max(d["keytot"], 1)
    pip_conv = bci(d["pipimp"], d["piptot"])
    one_on_one = bci(d["o2oheld"], d["o2osched"])
    training_hrs = min(d["trnhrs"] / max(d["trnhrstgt"], 1), 1.0)
    hp_retention = d["hpret"]
    team_eng = d["tmeng"] / 5.0
    career_plans = d["carplan"] / max(d["carplantgt"], 1)
    ramp = edp(max(0, d["rampdays"] - d["ramptgt"]), 0.05)
    perf_review = d["revcomp"]
    skills_gap = d["skgclosed"] / max(d["skgtot"], 1)
    X13 = hrc(
        values=[team_avg, team_delta, attrition, promo, succession,
                pip_conv, one_on_one, training_hrs, hp_retention, team_eng,
                career_plans, ramp, perf_review, skills_gap],
        weights=[0.10, 0.10, 0.08, 0.06, 0.06,
                 0.06, 0.08, 0.06, 0.08, 0.08,
                 0.06, 0.05, 0.06, 0.07]
    )

    d=team_manager_data[1]
    headcount = gaussian_penalty(d["prodhc"] / max(d["allochc"], 1), 1.0, 0.15)
    budget = edp(abs(d["tmbudact"] - d["tmbudalloc"])
                 / max(d["tmbudalloc"], 1), 3.0)
    cross_func = bci(d["cfdelot"], d["cfdeltot"])
    upward_fb = d["upfb"] / 5.0
    decision = edp(max(0, d["decdays"] - d["decdaysbench"]), 0.15)
    meetings = edp(d["mtghrs"], 5.0)
    sla = d["slamet"]
    capacity = d["capacc"]
    process = d["proccomp"]
    incident = edp(max(0, d["incresp"] - d["increspsla"]), 0.3)
    delegation = d["delgeff"]
    X14 = hrc(
        values=[headcount, budget, cross_func, upward_fb, decision,
                meetings, sla, capacity, process, incident, delegation],
        weights=[0.10, 0.08, 0.10, 0.14, 0.06,
                 0.06, 0.10, 0.08, 0.08, 0.10, 0.10]
    )

    d=team_manager_data[2]
    dei = d["tmdei"]
    psych_safety = d["psysafe"] / 5.0
    recognition = min(d["recogfreq"] / max(d["recogtgt"], 1), 1.0)
    conflict = d["confres"]
    change = d["chgadopt"]
    strategic = min(d["stratinit"] / max(d["stratinittgt"], 1), 1.0)
    innovation = ln_norm(d["innovideas"], 10)
    fb_loop = d["fbloop"]
    workload_eq = d["wleq"]
    X15 = hrc(
        values=[dei, psych_safety, recognition, conflict, change,
                strategic, innovation, fb_loop, workload_eq],
        weights=[0.10, 0.14, 0.10, 0.10, 0.10,
                 0.10, 0.10, 0.14, 0.12]
    )
    
elif role=="vendormanager":
    d=venodr_manager_data[0]
    sla_rate = bci(d["slasmet"], d["slastot"])
    cost_var = edp(max(0, (d["actvendspend"] - d["contrvendcost"])
                       / max(d["contrvendcost"], 1)), 4.0)
    renewal = bci(d["renfav"], d["rentot"])
    incidents = max(0, 1 - 0.20 * d["vendinc"])
    savings = min(d["savdel"] / max(d["savtgt"], 1), 1.3) / 1.3
    diversity = min(d["divspend"] / max(d["divspendtgt"], 0.01), 1.0)
    backup_cov = d["critcatcov"] / max(d["critcattot"], 1)
    quality = d["vendqual"]
    delivery = d["venddelot"]
    innovation_v = ln_norm(d["vendinnov"], 5)
    consolidation = min(d["vendcons"] / max(d["vendconstgt"], 1), 1.0)
    payment_opt = min(d["paytermopt"] / max(d["paytermtgt"], 1), 1.0)
    tco = min(d["tcored"] / 0.10, 1.0)
    X13 = hrc(
        values=[sla_rate, cost_var, renewal, incidents, savings,
                diversity, backup_cov, quality, delivery, innovation_v,
                consolidation, payment_opt, tco],
        weights=[0.12, 0.08, 0.08, 0.08, 0.10,
                 0.05, 0.07, 0.08, 0.08, 0.06,
                 0.06, 0.06, 0.08]
    )

    d=venodr_manager_data[1]
    contract_cov = d["spundcontr"] / max(d["totspend"], 1)
    rfp = bci(d["rfpcomp"], d["rfptot"])
    dd = d["venddd"] / max(d["vendact"], 1)
    po = bci(d["purchpo"], d["purchtot"])
    invoice_cycle = edp(max(0, d["invapprdays"] - d["invapprtgt"]), 0.3)
    risk_reg = d["vendriskass"] / max(d["vendrisktot"], 1)
    maverick = edp(d["mavspend"], 8.0)
    contract_expiry = d["contrexpmgd"]
    onboarding = edp(max(0, d["supponbdays"] - d["supponbtgt"]), 0.1)
    sustainability = d["sustcomp"]
    insurance = d["inscert"]
    X14 = hrc(
        values=[contract_cov, rfp, dd, po, invoice_cycle,
                risk_reg, maverick, contract_expiry, onboarding,
                sustainability, insurance],
        weights=[0.12, 0.10, 0.08, 0.10, 0.08,
                 0.08, 0.10, 0.08, 0.06,
                 0.10, 0.10]
    )

    d=venodr_manager_data[2]
    vendor_sat = d["vendsat"] / 5.0
    strategic_review = d["stratrev"]
    esc_resolved = bci(d["vendescres"], d["vendesctot"])
    market_intel = min(d["mktintel"] / max(d["mktinteltgt"], 1), 1.0)
    stakeholder_sat = d["stksat"] / 5.0
    dev_plans = min(d["venddevplan"] / max(d["venddevtgt"], 1), 1.0)
    cat_strategy = d["catstrat"]
    cross_func_s = ln_norm(d["cfsrcproj"], 5)
    legal_collab = d["legalcollab"]
    early_engage = d["earlyeng"]
    X15 = hrc(
        values=[vendor_sat, strategic_review, esc_resolved, market_intel,
                stakeholder_sat, dev_plans, cat_strategy, cross_func_s,
                legal_collab, early_engage],
        weights=[0.12, 0.10, 0.10, 0.08,
                 0.12, 0.08, 0.10, 0.08,
                 0.10, 0.12]
    )


In [19]:
def get_domain_config(role):
    if role == "sales":
        core_weight = 0.72
        domain_weights = {12: 0.12, 13: 0.09, 14: 0.07}
        extra_interactions = {
            (12, 0): +0.018,  
            (13, 1): +0.012,  
            (14, 3): +0.010,  
            (12, 13): -0.008, 
        }
        core_weight_overrides = {11: 0.070} 
    elif role == "projects":
        core_weight = 0.72
        domain_weights = {12: 0.12, 13: 0.09, 14: 0.07}
        extra_interactions = {
            (12, 13): +0.020, 
            (14, 12): -0.015,  
            (12, 1): +0.015, 
            (14, 3): +0.010,   
        }
        core_weight_overrides = {0: 0.135} 
    elif role == "hr":
        core_weight = 0.72
        domain_weights = {12: 0.12, 13: 0.09, 14: 0.07}
        extra_interactions = {
            (12, 13): +0.025,  
            (14, 9): +0.012,  
            (13, 2): +0.010, 
        }
        core_weight_overrides = {9: 0.060}
    elif role == "accounts":
        core_weight = 0.62  
        domain_weights = {12: 0.16, 13: 0.12, 14: 0.10}
        extra_interactions = {
            (12, 13): +0.020, 
            (14, 1): +0.012,   
            (12, 2): +0.010,   
        }
        core_weight_overrides = {1: 0.120} 
    elif role == "teammanager":
        core_weight = 0.68
        domain_weights = {12: 0.13, 13: 0.10, 14: 0.09}
        extra_interactions = {
            (12, 13): +0.030,  
            (12, 14): +0.015,  
            (13, 12): -0.020,  
            (14, 9): +0.012,   
        }
        core_weight_overrides = {8: 0.090}
    elif role == "vendor_manager":
        core_weight = 0.72
        domain_weights = {12: 0.12, 13: 0.09, 14: 0.07}
        extra_interactions = {
            (12, 14): +0.020,  
            (13, 12): -0.015,
            (14, 3): +0.010,   
            (12, 13): +0.012, 
        }
        core_weight_overrides = {}
    return core_weight, domain_weights, extra_interactions, core_weight_overrides

### GATES — Binary knockout conditions

Set to 't' (true/triggered) if the condition applies, 'f' (false) otherwise.
If ANY gate triggers, final score = 0.

G1: Employee is on active suspension, PIP with frozen evaluation, or formal disciplinary hold. 

G2: Employee has submitted resignation or been given termination notice.

G3: Role has been eliminated or is being made redundant.

G4: Employee is on extended leave (medical, parental, sabbatical)
    with insufficient data for a meaningful evaluation.
    
G5: Conflict of interest — employee is under investigation for fraud,
    embezzlement, or material policy violation that makes scoring premature.
    
G6: Data integrity failure — evaluation data has been flagged as
    fabricated, manipulated, or fundamentally unreliable.
    
G7: Legal hold — employee is party to active litigation against
    the organisation where scoring could create legal exposure.

In [20]:
g=[]
for i in range(len(gates)):
    h=gates[i]
    if h == 't':
        g.append(0)
    else:
        g.append(1)
G=1
for h in g:
    G=G*h
print(G)

1


### MODIFIERS — Continuous situational multipliers

M1: Data completeness confidence
    = 0.5 + 0.5 × (fraction of variables with confirmed data vs estimates)
    Range: [0.5, 1.0]
    

M2: Role transition adjustment

    1.0 = stable role for full evaluation period
    0.85 = changed roles/teams mid-period (partial data in new context)
    0.7 = brand new to the organisation (< 3 months)

M3: Evaluation period normalisation

    1.0 = full standard evaluation period completed
    0.85 = shortened period (e.g., mid-year hire evaluated at year-end)
    0.7 = significantly shortened (< 50% of standard period)

M4: External circumstance adjustment

    1.0 = normal operating conditions
    0.9 = team/org went through significant disruption (reorg, crisis)
    0.8 = severe external disruption (pandemic-level, office closure)

M5: Manager consistency

    1.0 = same manager for entire evaluation period
    0.9 = manager changed once mid-period
    0.8 = multiple manager changes or no consistent evaluato

In [21]:
a=[]
for i in range(len(modifiers)):
    h=modifiers[i]
    a.append(h)
M=1
for i in range(5):
    M=M*a[i]
print(M)

0.9


In [22]:
def sigmoid_transform(x, x0=0.50, k=8.0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))
def mu(S, a, a_ij):
    value = sum(a[i] for i in S)
    for (i, j), coeff in a_ij.items():
        if i in S and j in S:
            value += coeff
    return value
def choquet_integral(scores, a, a_ij):
    n = len(scores)
    sigma = sorted(range(n), key=lambda i: scores[i])
    integral = 0.0
    for rank in range(n):
        coalition = set(sigma[rank:])
        mu_val = mu(coalition, a, a_ij)
        if rank == 0:
            delta = scores[sigma[rank]]
        else:
            delta = scores[sigma[rank]] - scores[sigma[rank - 1]]
        integral += delta * mu_val
    return integral

In [26]:
core_weight, domain_weights, extra_interactions, core_overrides = get_domain_config(role)

X_all_raw = [X1, X2, X3, X4, X5, X6, X7, X8, X9, X10, X11, X12, X13, X14, X15]
sigmoid_index=[3,4,5,7,8,9]
X_all_sig = [sigmoid_transform(x) if i in sigmoid_index else x for i, x in enumerate(X_all_raw)]
X_final = [max(0.0, min(1.0, x)) for x in X_all_sig]
base_a = {
        0: 0.120, 1: 0.105, 2: 0.090, 3: 0.085, 4: 0.080,
        5: 0.070, 6: 0.065, 7: 0.080, 8: 0.055, 9: 0.045,
        10: 0.050, 11: 0.050
    }
for idx, new_w in core_overrides.items():
    base_a[idx] = new_w
core_sum = sum(base_a.values())
for k in base_a:
    base_a[k] = base_a[k] * (core_weight / core_sum)
a_vals = dict(base_a)
for k, w in domain_weights.items():
    a_vals[k] = w
base_interactions = {
        (0, 1): +0.025, (0, 2): +0.020, (4, 2): +0.015, (1, 6): +0.015,
        (4, 5): +0.015, (7, 11): +0.020, (8, 9): +0.010,
        (0, 6): -0.010, (3, 8): -0.010, (2, 10): -0.010, (5, 4): -0.005,
}
a_ij = {}
for k, v in base_interactions.items():
    a_ij[k] = v * core_weight
for k, v in extra_interactions.items():
    a_ij[k] = v
C_mu = choquet_integral(X_final, a_vals, a_ij)
mu_N = mu(set(range(15)), a_vals, a_ij)
S = 100.0 * G * M * (C_mu / mu_N)
print(S)

71.82050437671775
